# Neural Decoding Pipeline

Analysis code for *A Comparison of Neural Decoding Methods* (PIC, Paton Lab / Champalimaud).
Each section is a standalone script. Set the data paths (`data/`, `path/to/ephys_data`, `output/`) before running.


## 1. BG decoding pipeline — `s06_SVM_perSess.py`
Loops through mice and sessions, computes per-session decoding (SVM, LDA, Naive Bayes) for the Basal Ganglia, and saves results to `SVM_sess_BG.npz`.

In [ ]:
"""
s06_SVM_perSess.py
Loop through mice and sessions, compute spike counts with sliding window,
run SVM / LDA / Naive Bayes decoder (push/pull + DCnD directions).

Corrected structure:
  - ephys:    mouse/sess/eg_neurons_BG.mat          (h5py v7.3)
  - behavior: mouse/sess/behavior_fundamentals.mat  (scipy v5)
  - trial indices come from bhv struct in behavior file
  - spike times: eg_neurons['st_init'] shape (1, n_neu) object
                 → ref → (n_trials, 1) object
                 → ref → (1, n_spikes) float64
"""

import os
import time
import numpy as np
import scipy.io as sio
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from decoder_mc_cv import decode


# ===========================================================================
# Loaders
# ===========================================================================

def _load_behavior(filepath):
    """Load behavior_fundamentals.mat (scipy v5), return bhv dict."""
    data = sio.loadmat(filepath, simplify_cells=True)
    return data['bhv']


def _arr(val):
    """Flatten any nested array to a 1D numpy array."""
    return np.atleast_1d(np.squeeze(np.asarray(val))).ravel()


def _get_n_neu(filepath):
    """Return number of neurons from eg_neurons_BG.mat without loading data."""
    with h5py.File(filepath, 'r') as f:
        return f['eg_neurons']['st_init'].shape[1]


def _load_spike_times(filepath, n_neu):
    """
    Load st_init and st_reach efficiently from eg_neurons_BG.mat (h5py v7.3).

    Structure in HDF5:
      eg_neurons['st_init']  shape (1, n_neu) object references
        → each ref → Dataset shape (n_trials, 1) object references
          → each ref → Dataset shape (1, n_spikes) float64

    Returns
    -------
    st_init, st_reach : list of length n_neu
        Each element is a list of n_trials 1D spike-time arrays.
    """
    st_init  = []
    st_reach = []

    with h5py.File(filepath, 'r') as f:
        eg = f['eg_neurons']

        for n in range(n_neu):
            ds_init  = f[eg['st_init'][0, n]]    # (n_trials, 1) object
            ds_reach = f[eg['st_reach'][0, n]]

            spk_init  = [f[ds_init[tt, 0]][()].ravel().astype(float)
                         for tt in range(ds_init.shape[0])]
            spk_reach = [f[ds_reach[tt, 0]][()].ravel().astype(float)
                         for tt in range(ds_reach.shape[0])]

            st_init.append(spk_init)
            st_reach.append(spk_reach)

    return st_init, st_reach


# ===========================================================================
# Paths and configuration
# ===========================================================================
rootdir  = r'path/to/ephys_data'  # ephys_and_behavior/mat_files directory
group    = '20230801_ChocolateGroup'
reg      = 'BG'
base_mat = os.path.join(rootdir, 'ephys_and_behavior', 'mat_files', group)
os.makedirs(base_mat, exist_ok=True)

animals  = ['CoteDor', 'Lindt', 'Toblerone', 'Milka', 'FerreroRocher']
# All real sessions per animal — sessions without eg_neurons_BG.mat are skipped
all_sess = [
    ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7'],
    ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7'],
    ['R1', 'R2', 'R3', 'R4', 'R5', 'R6'],
    ['R1', 'R4', 'R5', 'R6', 'R7'],
    ['R1', 'R4', 'R5', 'R6'],
]

# ===========================================================================
# Sliding-window parameters
# ===========================================================================
win_int   = [-2.5, 2.5]
tm_before, tm_after = win_int
win_dur   = tm_after - tm_before
bin_width = 0.1
bin_step  = 0.025
n_bins    = int(np.floor((win_dur / bin_step) - (bin_width / bin_step)) + 1)

bin_edges = np.array([
    [tm_before + i * bin_step,
     tm_before + bin_width + i * bin_step]
    for i in range(n_bins)
])
bin_mean = bin_edges[:, 0] + bin_width / 2

# ===========================================================================
# Figure colours
# ===========================================================================
clrs        = [np.array([63,130,109])/256, np.array([48,131,220])/256,
               np.array([72,60,50])/256,   np.array([158,216,219])/256]
labels_plot = ['push/pull', 'dominant', 'center', 'non-dominant']

# ===========================================================================
# Decoder parameters
# ===========================================================================
ratio_train_val = 0.8
ncv             = 100
nfold           = 10
Cvec            = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
METHODS         = ['svm', 'lda', 'nb']
N_JOBS_BINS     = -1
N_JOBS_CV       = 1


# ===========================================================================
# Worker: one time bin × all contrasts × all decoders
# ===========================================================================
def _process_bin(b, spk_count_init, spk_count_reach,
                 i_pp, i_DCnD, r_pp, r_hit, r_DCnD,
                 ratio_train_val, ncv, nfold, Cvec, methods, n_jobs_cv):

    kw_base = dict(ratio_train_val=ratio_train_val, ncv=ncv,
                   nfold=nfold, Cvec=Cvec, n_jobs=n_jobs_cv)
    sc_i = spk_count_init[b]
    sc_r = spk_count_reach[b]
    hit  = r_hit == 1

    pairs_init = {
        'pp': (sc_i[i_pp == 0, :],                           sc_i[i_pp == 1, :]),
        'D':  (sc_i[i_DCnD == 1, :],                         sc_i[(i_DCnD==2)|(i_DCnD==3), :]),
        'C':  (sc_i[i_DCnD == 2, :],                         sc_i[(i_DCnD==1)|(i_DCnD==3), :]),
        'nD': (sc_i[i_DCnD == 3, :],                         sc_i[(i_DCnD==1)|(i_DCnD==2), :]),
    }
    pairs_reach = {
        'pp': (sc_r[(r_pp==0)&hit, :],                       sc_r[(r_pp==1)&hit, :]),
        'D':  (sc_r[(r_DCnD==1)&hit, :],                     sc_r[((r_DCnD==2)|(r_DCnD==3))&hit, :]),
        'C':  (sc_r[(r_DCnD==2)&hit, :],                     sc_r[((r_DCnD==1)|(r_DCnD==3))&hit, :]),
        'nD': (sc_r[(r_DCnD==3)&hit, :],                     sc_r[((r_DCnD==1)|(r_DCnD==2))&hit, :]),
    }

    out = {}
    for method in methods:
        kw = dict(**kw_base, method=method)
        for contrast, (s1, s2) in pairs_init.items():
            out[f'{method}_{contrast}_init']  = decode(s1, s2, **kw)
        for contrast, (s1, s2) in pairs_reach.items():
            out[f'{method}_{contrast}_reach'] = decode(s1, s2, **kw)
    return out


# ===========================================================================
# Main loop — mice × sessions
# ===========================================================================
SVM_sess = []
cc = 0

for m, animal in enumerate(animals):
    mouse = f'{m+1}_{animal}'
    print(f'\nLooping through {mouse}:')

    for sess in all_sess[m]:
        sess_base  = os.path.join(base_mat, mouse, sess)
        ephys_path = os.path.join(sess_base, f'eg_neurons_{reg}.mat')
        behav_path = os.path.join(sess_base, 'behavior_fundamentals.mat')

        if not os.path.isfile(ephys_path):
            print(f'  {sess}: no {reg} data — skipping')
            continue
        if not os.path.isfile(behav_path):
            print(f'  {sess}: no behavior file — skipping')
            continue

        print(f'  Running {mouse} {sess}...')
        cc += 1

        # ── Behavior indices ─────────────────────────────────────────────────
        bhv = _load_behavior(behav_path)

        # ── Init trial indices (313 trials) ────────────────────────────────
        # init_invPush_invPull_idx: 1=inv-Push, 2=inv-Pull, 3=no PP — len=313
        # DCnD_init              : 0=none, 1=D, 2=C, 3=nD            — len=313
        # Push/pull in init = invalid push vs invalid pull (animal initiated
        # but did not complete the movement). Recoded to 0=inv-Push, 1=inv-Pull.
        inv_pp = _arr(bhv['init_invPush_invPull_idx']).astype(int)  # 1/2/3
        i_pp   = np.where(inv_pp == 1, 0,
                 np.where(inv_pp == 2, 1, 3))  # 0=invPush, 1=invPull, 3=other

        i_DCnD = _arr(bhv['DCnD_init']).astype(int)   # 0=none,1=D,2=C,3=nD

        # ── Reach trial indices (565 trials) ────────────────────────────────
        # pp_inVec_idx : 0=push, 1=pull — valid completed push/pull
        # DCnD_inVec_idx: 1=D, 2=C, 3=nD
        # hit_inVec    : 0/1, has NaNs (NaN → 0 = miss)
        r_pp   = _arr(bhv['pp_inVec_idx']).astype(int)       # 0=push, 1=pull
        r_DCnD = _arr(bhv['DCnD_inVec_idx']).astype(int)     # 1=D,2=C,3=nD

        hit_raw = np.atleast_1d(bhv['hit_inVec']).ravel().astype(float)
        r_hit   = np.nan_to_num(hit_raw, nan=0.0).astype(int)  # NaN → 0

        n_trials_pp = len(i_pp)    # 313 init trials
        n_trials_rr = len(r_hit)   # 565 reach trials

        # ── Spike times ──────────────────────────────────────────────────────
        n_neu = _get_n_neu(ephys_path)
        print(f'    {n_neu} neurons | {n_trials_pp} init trials | {n_trials_rr} reach trials')
        st_init, st_reach = _load_spike_times(ephys_path, n_neu)

        # ── Spike counts — sliding window ────────────────────────────────────
        spk_count_init  = np.zeros((n_bins, n_trials_pp, n_neu), dtype=np.float32)
        spk_count_reach = np.zeros((n_bins, n_trials_rr, n_neu), dtype=np.float32)

        for n_idx in range(n_neu):
            for tt in range(n_trials_pp):
                spk = st_init[n_idx][tt]
                for i, (bs, be) in enumerate(bin_edges):
                    spk_count_init[i, tt, n_idx] = np.sum((spk > bs) & (spk < be))
            for tt in range(n_trials_rr):
                spk = st_reach[n_idx][tt]
                for i, (bs, be) in enumerate(bin_edges):
                    spk_count_reach[i, tt, n_idx] = np.sum((spk > bs) & (spk < be))

        # ── Decoders ─────────────────────────────────────────────────────────
        print(f'    Decoding {METHODS} across {n_bins} bins...')
        t0 = time.time()

        results = Parallel(n_jobs=N_JOBS_BINS, verbose=0)(
            delayed(_process_bin)(
                b, spk_count_init, spk_count_reach,
                i_pp, i_DCnD, r_pp, r_hit, r_DCnD,
                ratio_train_val, ncv, nfold, Cvec, METHODS, N_JOBS_CV
            )
            for b in range(n_bins)
        )
        print(f'    done!!  {time.time() - t0:.1f}s')

        contrasts = ['pp', 'D', 'C', 'nD']
        bac = {method: {c: np.array([[r[f'{method}_{c}_init'],
                                       r[f'{method}_{c}_reach']]
                                      for r in results])
                        for c in contrasts}
               for method in METHODS}

        # ── Figures ──────────────────────────────────────────────────────────
        save_out_dir = os.path.join(
            r'.', mouse, sess
        )
        os.makedirs(save_out_dir, exist_ok=True)

        for method in METHODS:
            fig, axes = plt.subplots(1, 2, figsize=(13, 4))
            fig.patch.set_facecolor('white')
            fig.suptitle(f'{mouse}  |  {sess}  |  {method.upper()}', fontsize=11)
            for ax, col_idx, xlabel in zip(
                axes, [0, 1],
                ['time from trial init (s)', 'time from reach endpoint (s)']
            ):
                for pi, contrast in enumerate(contrasts):
                    ax.plot(bin_mean, bac[method][contrast][:, col_idx],
                            color=clrs[pi], linewidth=2, label=labels_plot[pi])
                ax.axhline(0.5, linestyle='--', color=[.7,.7,.7], linewidth=1, alpha=0.7)
                ax.set_xlim(win_int); ax.set_ylim([0.45, 1])
                ax.set_xlabel(xlabel, fontsize=10); ax.set_ylabel('accuracy', fontsize=10)
                ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
                ax.tick_params(direction='out', length=3, width=1.5)
            axes[1].legend(frameon=False, fontsize=9)
            plt.tight_layout()
            fig.savefig(os.path.join(save_out_dir, f'decoder_{method}_accuracy.png'), dpi=150)
            plt.close(fig)

        # ── Store ─────────────────────────────────────────────────────────────
        SVM_sess.append({
            'mouse': mouse, 'sess': sess, 'n_neu': n_neu,
            'bin_edges': bin_edges, 'bac': bac,
            'params': dict(win_int=win_int, ratio_train_val=ratio_train_val,
                           ncv=ncv, nfold=nfold, Cvec=Cvec,
                           bin_width=bin_width, bin_step=bin_step, methods=METHODS),
        })

# ===========================================================================
# Save
# ===========================================================================
print('\nSaving...')
out_path = os.path.join(r'.', f'SVM_sess_{reg}.npz')
np.savez(out_path, SVM_sess=np.array(SVM_sess, dtype=object), allow_pickle=True)
print(f'All done!!  {cc} sessions processed.  Saved to {out_path}')

## 2. CB decoding pipeline — `s06_SVM_perSess_CB.py`
Same pipeline for the Cerebellum, with per-cell-type handling; saves to `SVM_sess_CB.npz`.

In [ ]:
"""
s06_SVM_perSess_CB.py
=====================
Loop through mice and sessions for the CEREBELLUM (CB) region.
Identical pipeline to s06_SVM_perSess.py (BG) but additionally:
  - Loads cell_type_CB.mat (scipy v5) to get cell type per neuron
  - Filters out invalid neurons (cb_isInValid == 1)
  - Saves cell_type per neuron alongside BAC results
  - Allows running decoder separately per cell type

Cell types found in cb_cell_type[n]['cell_type']:
  'PkC_cs' — Purkinje cell (complex spikes)
  'GoC'    — Golgi cell
  'MFB'    — Mossy Fibre Bouton
  'NaN'    — unclassified

Files per session:
  eg_neurons_CB.mat   — v7.3 (h5py): spike times, same structure as BG
  cell_type_CB.mat    — v5  (scipy): cell type labels + validity flag
"""

import os
import time
import numpy as np
import scipy.io as sio
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from decoder_mc_cv import decode


# ===========================================================================
# Loaders  (same as BG version)
# ===========================================================================

def _load_behavior(filepath):
    data = sio.loadmat(filepath, simplify_cells=True)
    return data['bhv']

def _arr(val):
    return np.atleast_1d(np.squeeze(np.asarray(val))).ravel()

def _load_cell_types(filepath):
    """
    Load cell_type_CB.mat (scipy v5).

    Returns
    -------
    cell_types : list of str, length n_neu
        Cell type label per neuron ('PkC_cs', 'GoC', 'MFB', 'NaN').
    is_valid : np.ndarray of bool, length n_neu
        True if neuron is valid (cb_isInValid == 0).
    """
    data        = sio.loadmat(filepath, simplify_cells=True)
    cb_ct       = data['cb_cell_type']        # array of dicts, one per neuron
    cb_invalid  = _arr(data['cb_isInValid'])  # 0=valid, 1=invalid

    cell_types = []
    for entry in cb_ct:
        ct = entry.get('cell_type', 'NaN')
        # Sometimes stored as array([]) for unclassified
        if not isinstance(ct, str):
            ct = 'NaN'
        cell_types.append(ct)

    is_valid = (cb_invalid == 0)
    return cell_types, is_valid

def _get_n_neu(filepath):
    with h5py.File(filepath, 'r') as f:
        return f['eg_neurons']['st_init'].shape[1]

def _load_spike_times(filepath, n_neu):
    """Same HDF5 loader as BG version."""
    st_init  = []
    st_reach = []
    with h5py.File(filepath, 'r') as f:
        eg = f['eg_neurons']
        for n in range(n_neu):
            ds_init  = f[eg['st_init'][0, n]]
            ds_reach = f[eg['st_reach'][0, n]]
            spk_init  = [f[ds_init[tt, 0]][()].ravel().astype(float)
                         for tt in range(ds_init.shape[0])]
            spk_reach = [f[ds_reach[tt, 0]][()].ravel().astype(float)
                         for tt in range(ds_reach.shape[0])]
            st_init.append(spk_init)
            st_reach.append(spk_reach)
    return st_init, st_reach


# ===========================================================================
# Paths and configuration
# ===========================================================================
rootdir  = r'path/to/ephys_data'  # ephys_and_behavior/mat_files directory
group    = '20230801_ChocolateGroup'
reg      = 'CB'
base_mat = os.path.join(rootdir, 'ephys_and_behavior', 'mat_files', group)
os.makedirs(base_mat, exist_ok=True)

animals  = ['CoteDor', 'Lindt', 'Toblerone', 'Milka', 'FerreroRocher']
all_sess = [
    ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7'],
    ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7'],
    ['R1', 'R2', 'R3', 'R4', 'R5', 'R6'],
    ['R1', 'R4', 'R5', 'R6', 'R7'],
    ['R1', 'R4', 'R5', 'R6'],
]

# Cell types to decode separately (in addition to ALL neurons combined)
# Set to None to skip per-cell-type decoding
CELL_TYPES_DECODE = ['PkC_cs', 'GoC', 'MFB', 'all']

# ===========================================================================
# Sliding-window parameters
# ===========================================================================
win_int   = [-2.5, 2.5]
tm_before, tm_after = win_int
win_dur   = tm_after - tm_before
bin_width = 0.1
bin_step  = 0.025
n_bins    = int(np.floor((win_dur / bin_step) - (bin_width / bin_step)) + 1)

bin_edges = np.array([
    [tm_before + i * bin_step,
     tm_before + bin_width + i * bin_step]
    for i in range(n_bins)
])
bin_mean = bin_edges[:, 0] + bin_width / 2

# ===========================================================================
# Figure colours
# ===========================================================================
clrs        = [np.array([63,130,109])/256, np.array([48,131,220])/256,
               np.array([72,60,50])/256,   np.array([158,216,219])/256]
labels_plot = ['push/pull', 'dominant', 'center', 'non-dominant']

# Cell type colours for per-type plots
CT_COLORS = {
    'PkC_cs': np.array([220, 50,  50]) / 255,
    'GoC':    np.array([50,  50, 220]) / 255,
    'MFB':    np.array([50, 180,  50]) / 255,
    'all':    np.array([80,  80,  80]) / 255,
}

# ===========================================================================
# Decoder parameters
# ===========================================================================
ratio_train_val = 0.8
ncv             = 100
nfold           = 10
Cvec            = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
METHODS         = ['svm', 'lda', 'nb']
N_JOBS_BINS     = -1
N_JOBS_CV       = 1
MIN_TRIALS      = 10


def _safe_decode(s1, s2, min_trials, **kw):
    if s1.shape[0] < min_trials or s2.shape[0] < min_trials:
        return np.nan
    return decode(s1, s2, **kw)


# ===========================================================================
# Worker: one time bin × all contrasts × all decoders × all cell-type subsets
# ===========================================================================
def _process_bin(b, spk_count_init, spk_count_reach,
                 i_pp, i_DCnD, r_pp, r_hit, r_DCnD,
                 neuron_masks,           # dict: ct_label -> bool array (n_neu,)
                 ratio_train_val, ncv, nfold, Cvec, methods, n_jobs_cv,
                 min_trials=MIN_TRIALS):

    kw_base = dict(ratio_train_val=ratio_train_val, ncv=ncv,
                   nfold=nfold, Cvec=Cvec, n_jobs=n_jobs_cv)
    sc_i = spk_count_init[b]    # (n_trials_pp, n_neu_valid)
    sc_r = spk_count_reach[b]   # (n_trials_rr, n_neu_valid)
    hit  = r_hit == 1

    # Trial splits — same as BG
    pairs_init = {
        'pp': (sc_i[i_pp == 0, :],              sc_i[i_pp == 1, :]),
        'D':  (sc_i[i_DCnD == 1, :],            sc_i[(i_DCnD==2)|(i_DCnD==3), :]),
        'C':  (sc_i[i_DCnD == 2, :],            sc_i[(i_DCnD==1)|(i_DCnD==3), :]),
        'nD': (sc_i[i_DCnD == 3, :],            sc_i[(i_DCnD==1)|(i_DCnD==2), :]),
    }
    pairs_reach = {
        'pp': (sc_r[(r_pp==0)&hit, :],          sc_r[(r_pp==1)&hit, :]),
        'D':  (sc_r[(r_DCnD==1)&hit, :],        sc_r[((r_DCnD==2)|(r_DCnD==3))&hit, :]),
        'C':  (sc_r[(r_DCnD==2)&hit, :],        sc_r[((r_DCnD==1)|(r_DCnD==3))&hit, :]),
        'nD': (sc_r[(r_DCnD==3)&hit, :],        sc_r[((r_DCnD==1)|(r_DCnD==2))&hit, :]),
    }

    out = {}
    for ct_label, neu_mask in neuron_masks.items():
        for method in methods:
            kw = dict(**kw_base, method=method)
            for contrast, (s1_all, s2_all) in pairs_init.items():
                s1 = s1_all[:, neu_mask]
                s2 = s2_all[:, neu_mask]
                key = f'{ct_label}_{method}_{contrast}_init'
                out[key] = _safe_decode(s1, s2, min_trials, **kw)
            for contrast, (s1_all, s2_all) in pairs_reach.items():
                s1 = s1_all[:, neu_mask]
                s2 = s2_all[:, neu_mask]
                key = f'{ct_label}_{method}_{contrast}_reach'
                out[key] = _safe_decode(s1, s2, min_trials, **kw)
    return out


# ===========================================================================
# Main loop — mice × sessions
# ===========================================================================
SVM_sess = []
cc = 0

for m, animal in enumerate(animals):
    mouse = f'{m+1}_{animal}'
    print(f'\nLooping through {mouse}:')

    for sess in all_sess[m]:
        sess = sess.strip()
        sess_base    = os.path.join(base_mat, mouse, sess)
        ephys_path   = os.path.join(sess_base, f'eg_neurons_{reg}.mat')
        ct_path      = os.path.join(sess_base, f'cell_type_{reg}.mat')
        behav_path   = os.path.join(sess_base, 'behavior_fundamentals.mat')

        if not os.path.isfile(ephys_path):
            print(f'  {sess}: no {reg} ephys — skipping')
            continue
        if not os.path.isfile(behav_path):
            print(f'  {sess}: no behavior — skipping')
            continue

        print(f'  Running {mouse} {sess}...')
        cc += 1

        # ── Behavior ─────────────────────────────────────────────────────────
        bhv = _load_behavior(behav_path)

        inv_pp = _arr(bhv['init_invPush_invPull_idx']).astype(int)
        i_pp   = np.where(inv_pp == 1, 0, np.where(inv_pp == 2, 1, 3))
        i_DCnD = _arr(bhv['DCnD_init']).astype(int)
        r_pp   = _arr(bhv['pp_inVec_idx']).astype(int)
        r_DCnD = _arr(bhv['DCnD_inVec_idx']).astype(int)
        hit_raw = np.atleast_1d(bhv['hit_inVec']).ravel().astype(float)
        r_hit   = np.nan_to_num(hit_raw, nan=0.0).astype(int)

        n_trials_pp = len(i_pp)
        n_trials_rr = len(r_hit)

        # ── Cell types ───────────────────────────────────────────────────────
        n_neu_total = _get_n_neu(ephys_path)

        if os.path.isfile(ct_path):
            cell_types_all, is_valid_all = _load_cell_types(ct_path)
            # Ensure lengths match (cell_type file may have more/fewer entries)
            if len(cell_types_all) != n_neu_total:
                print(f'    WARNING: cell_type len {len(cell_types_all)} '
                      f'!= n_neu {n_neu_total} — using all neurons as valid')
                cell_types_all = ['NaN'] * n_neu_total
                is_valid_all   = np.ones(n_neu_total, dtype=bool)
        else:
            print(f'    No cell_type_{reg}.mat — treating all neurons as valid')
            cell_types_all = ['NaN'] * n_neu_total
            is_valid_all   = np.ones(n_neu_total, dtype=bool)

        # Keep only valid neurons
        valid_idx   = np.where(is_valid_all)[0]
        n_neu_valid = len(valid_idx)
        cell_types  = [cell_types_all[i] for i in valid_idx]

        print(f'    {n_neu_total} total neurons | {n_neu_valid} valid | '
              f'{n_trials_pp} init trials | {n_trials_rr} reach trials')
        ct_counts = {ct: cell_types.count(ct) for ct in set(cell_types)}
        print(f'    Cell types: {ct_counts}')

        # ── Spike times (valid neurons only) ─────────────────────────────────
        st_init_all, st_reach_all = _load_spike_times(ephys_path, n_neu_total)
        st_init  = [st_init_all[i]  for i in valid_idx]
        st_reach = [st_reach_all[i] for i in valid_idx]

        # ── Spike counts — sliding window ────────────────────────────────────
        spk_count_init  = np.zeros((n_bins, n_trials_pp, n_neu_valid), dtype=np.float32)
        spk_count_reach = np.zeros((n_bins, n_trials_rr, n_neu_valid), dtype=np.float32)

        for n_idx in range(n_neu_valid):
            for tt in range(n_trials_pp):
                spk = np.atleast_1d(st_init[n_idx][tt]).ravel()
                for i, (bs, be) in enumerate(bin_edges):
                    spk_count_init[i, tt, n_idx] = np.sum((spk > bs) & (spk < be))
            for tt in range(n_trials_rr):
                spk = np.atleast_1d(st_reach[n_idx][tt]).ravel()
                for i, (bs, be) in enumerate(bin_edges):
                    spk_count_reach[i, tt, n_idx] = np.sum((spk > bs) & (spk < be))

        # ── Build neuron masks per cell type ─────────────────────────────────
        ct_arr = np.array(cell_types)
        neuron_masks = {}
        for ct_label in CELL_TYPES_DECODE:
            if ct_label == 'all':
                mask = np.ones(n_neu_valid, dtype=bool)
            else:
                mask = (ct_arr == ct_label)
            if mask.sum() == 0:
                print(f'    No neurons of type {ct_label} — skipping')
                continue
            print(f'    Cell type {ct_label}: {mask.sum()} neurons')
            neuron_masks[ct_label] = mask

        # ── Decoders ─────────────────────────────────────────────────────────
        print(f'    Decoding {METHODS} × {list(neuron_masks.keys())} '
              f'across {n_bins} bins...')
        t0 = time.time()

        results = Parallel(n_jobs=N_JOBS_BINS, verbose=0)(
            delayed(_process_bin)(
                b, spk_count_init, spk_count_reach,
                i_pp, i_DCnD, r_pp, r_hit, r_DCnD,
                neuron_masks,
                ratio_train_val, ncv, nfold, Cvec, METHODS, N_JOBS_CV
            )
            for b in range(n_bins)
        )
        print(f'    done!!  {time.time() - t0:.1f}s')

        # Unpack: bac[ct_label][method][contrast] = (n_bins, 2) array
        contrasts = ['pp', 'D', 'C', 'nD']
        bac = {}
        for ct_label in neuron_masks:
            bac[ct_label] = {}
            for method in METHODS:
                bac[ct_label][method] = {}
                for contrast in contrasts:
                    ki = f'{ct_label}_{method}_{contrast}_init'
                    kr = f'{ct_label}_{method}_{contrast}_reach'
                    bac[ct_label][method][contrast] = np.array([
                        [r.get(ki, np.nan), r.get(kr, np.nan)]
                        for r in results
                    ])  # (n_bins, 2)

        # ── Figures ──────────────────────────────────────────────────────────
        save_out_dir = os.path.join(
            r'.', mouse, sess
        )
        os.makedirs(save_out_dir, exist_ok=True)

        for ct_label in neuron_masks:
            for method in METHODS:
                fig, axes = plt.subplots(1, 2, figsize=(13, 4))
                fig.patch.set_facecolor('white')
                n_ct = neuron_masks[ct_label].sum()
                fig.suptitle(f'{mouse}  |  {sess}  |  CB  |  {method.upper()}'
                             f'  |  {ct_label} (n={n_ct})', fontsize=10)

                for ax, col_idx, xlabel in zip(
                    axes, [0, 1],
                    ['time from trial init (s)', 'time from reach endpoint (s)']
                ):
                    for pi, contrast in enumerate(contrasts):
                        ax.plot(bin_mean,
                                bac[ct_label][method][contrast][:, col_idx],
                                color=clrs[pi], linewidth=2,
                                label=labels_plot[pi])
                    ax.axhline(0.5, linestyle='--', color=[.7,.7,.7],
                               linewidth=1, alpha=0.7)
                    ax.set_xlim(win_int); ax.set_ylim([0.45, 1])
                    ax.set_xlabel(xlabel, fontsize=10)
                    ax.set_ylabel('accuracy', fontsize=10)
                    ax.spines['top'].set_visible(False)
                    ax.spines['right'].set_visible(False)
                    ax.tick_params(direction='out', length=3, width=1.5)

                axes[1].legend(frameon=False, fontsize=9)
                plt.tight_layout()
                fname = f'decoder_{method}_{ct_label}_accuracy.png'
                fig.savefig(os.path.join(save_out_dir, fname), dpi=150)
                plt.close(fig)

        # ── Store ─────────────────────────────────────────────────────────────
        SVM_sess.append({
            'mouse':       mouse,
            'sess':        sess,
            'n_neu_total': n_neu_total,
            'n_neu_valid': n_neu_valid,
            'cell_types':  cell_types,       # list of str, one per valid neuron
            'neuron_masks': {k: v.tolist() for k, v in neuron_masks.items()},
            'bin_edges':   bin_edges,
            'bac':         bac,
            'params': dict(
                win_int=win_int, ratio_train_val=ratio_train_val,
                ncv=ncv, nfold=nfold, Cvec=Cvec,
                bin_width=bin_width, bin_step=bin_step,
                methods=METHODS, cell_types_decode=CELL_TYPES_DECODE,
            ),
        })

# ===========================================================================
# Save
# ===========================================================================
print('\nSaving...')
out_path = os.path.join(
    r'.', f'SVM_sess_{reg}.npz'
)
np.savez(out_path, SVM_sess=np.array(SVM_sess, dtype=object),
         allow_pickle=True)
print(f'All done!!  {cc} sessions processed.  Saved to {out_path}')

## 3. Add Kalman decoder — `add_kalman.py`
Adds the adapted binary Kalman filter decoder to the existing `.npz` results (self-contained, thread-based parallelism).

In [ ]:
"""
add_kalman.py
=============
Adiciona o decoder de Kalman filter aos npz existentes (SVM_sess_BG.npz
e SVM_sess_CB.npz) sem re-correr SVM, LDA ou NB.

Como funciona:
  1. Carrega o npz existente
  2. Para cada sessão, re-carrega os ficheiros .mat para obter spike counts
  3. Corre APENAS o Kalman decoder (sem inner CV → muito mais rápido)
  4. Insere os resultados em sess['bac']['kalman'] e guarda o npz actualizado

Velocidade esperada: ~5-10x mais rápido que SVM por sessão (sem inner CV,
sem iteração de optimização — apenas operações matriciais).
"""

import os
import time
import numpy as np
import scipy.io as sio
import h5py
from joblib import Parallel, delayed

# ===========================================================================
# Kalman decoder — self-contained, no external imports needed
# ===========================================================================

def _balanced_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    classes = np.unique(y_true)
    if len(classes) < 2:
        return float('nan')
    tp = np.sum((y_pred == 1) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    sens = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    spec = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
    if np.isnan(sens):
        return spec
    if np.isnan(spec):
        return sens
    return (sens + spec) / 2.0


def _kalman_one_cv(s1n, s2n, n1, n2, ntrain1, ntrain2):
    try:
        rp1 = np.random.permutation(n1)
        rp2 = np.random.permutation(n2)
        tr1, va1 = rp1[:ntrain1], rp1[ntrain1:]
        tr2, va2 = rp2[:ntrain2], rp2[ntrain2:]

        X_tr  = np.vstack([s1n[tr1], s2n[tr2]])
        X_val = np.vstack([s1n[va1], s2n[va2]])
        y_val = np.array([0]*len(va1) + [1]*len(va2))

        # Observation model
        mu1 = np.nanmean(s1n[tr1], axis=0)
        mu2 = np.nanmean(s2n[tr2], axis=0)
        C   = (mu1 - mu2) / 2.0

        var1  = np.nanvar(s1n[tr1], axis=0, ddof=1)
        var2  = np.nanvar(s2n[tr2], axis=0, ddof=1)
        R     = ((ntrain1-1)*var1 + (ntrain2-1)*var2) / (ntrain1+ntrain2-2)
        R     = np.where(R == 0, 1e-6, R)

        CRinv = C / R
        denom = np.dot(CRinv, C) + 1.0
        if denom == 0:
            return float('nan')

        x_hat = X_val @ CRinv / denom
        pred  = (x_hat < 0).astype(int)   # +1 → class 0 (s1), -1 → class 1 (s2)
        return _balanced_accuracy(y_val, pred)
    except Exception:
        return float('nan')


def kalman_decode(s1, s2, ratio_train_val=0.8, ncv=100, n_jobs=1):
    s1 = np.asarray(s1, dtype=float)
    s2 = np.asarray(s2, dtype=float)
    n1, n2 = s1.shape[0], s2.shape[0]
    if n1 < 2 or n2 < 2:
        return float('nan')

    # Normalise with train stats (approximate — use full set for mean/max)
    mat  = np.vstack([s1, s2])
    mean = np.mean(mat, axis=0)
    mx   = np.max(mat, axis=0)
    denom = np.where(mx == 0, 1.0, mx)
    s1n  = (s1 - mean) / denom
    s2n  = (s2 - mean) / denom

    ntrain1 = int(np.floor(n1 * ratio_train_val))
    ntrain2 = int(np.floor(n2 * ratio_train_val))
    if ntrain1 < 2 or ntrain2 < 2:
        return float('nan')

    bac_cv = Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(_kalman_one_cv)(s1n, s2n, n1, n2, ntrain1, ntrain2)
        for _ in range(ncv)
    )
    return float(np.nanmean(bac_cv))

# ===========================================================================
# Configuration — igual ao perSess original
# ===========================================================================
rootdir  = r'path/to/ephys_data'  # ephys_and_behavior/mat_files directory
group    = '20230801_ChocolateGroup'
base_mat = os.path.join(rootdir, 'ephys_and_behavior', 'mat_files', group)

BG_NPZ = r'data/SVM_sess_BG.npz'
CB_NPZ = r'data/SVM_sess_CB.npz'

animals  = ['CoteDor', 'Lindt', 'Toblerone', 'Milka', 'FerreroRocher']
all_sess = [
    ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7'],
    ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7'],
    ['R1', 'R2', 'R3', 'R4', 'R5', 'R6'],
    ['R1', 'R4', 'R5', 'R6', 'R7'],
    ['R1', 'R4', 'R5', 'R6'],
]

# CB cell types to decode (must match what was used in s06_SVM_perSess_CB)
CB_CELL_TYPES = ['PkC_cs', 'PkC_ss', 'GoC', 'MFB', 'MLI', 'all']

# Decoder parameters
ratio_train_val = 0.8
ncv             = 100
N_JOBS_BINS     = -1
N_JOBS_CV       = 1
MIN_TRIALS      = 10

# Sliding window (must match perSess)
win_int   = [-2.5, 2.5]
bin_width = 0.1
bin_step  = 0.025
tm_before, tm_after = win_int
win_dur   = tm_after - tm_before
n_bins    = int(np.floor((win_dur / bin_step) - (bin_width / bin_step)) + 1)
bin_edges = np.array([
    [tm_before + i * bin_step,
     tm_before + bin_width + i * bin_step]
    for i in range(n_bins)
])

CONTRASTS = ['pp', 'D', 'C', 'nD']


# ===========================================================================
# Shared loaders (copied from perSess scripts)
# ===========================================================================

def _arr(val):
    return np.atleast_1d(np.squeeze(np.asarray(val))).ravel()

def _load_behavior(filepath):
    data = sio.loadmat(filepath, simplify_cells=True)
    return data['bhv']

def _load_spike_times_BG(filepath, n_neu):
    st_init, st_reach = [], []
    with h5py.File(filepath, 'r') as f:
        eg = f['eg_neurons']
        for n in range(n_neu):
            ds_i = f[eg['st_init'][0, n]]
            ds_r = f[eg['st_reach'][0, n]]
            st_init.append([f[ds_i[tt, 0]][()].ravel().astype(float)
                            for tt in range(ds_i.shape[0])])
            st_reach.append([f[ds_r[tt, 0]][()].ravel().astype(float)
                             for tt in range(ds_r.shape[0])])
    return st_init, st_reach

def _n_neu(filepath):
    with h5py.File(filepath, 'r') as f:
        return f['eg_neurons']['st_init'].shape[1]

def _read_hdf5_str(ds, f):
    try:
        data = ds[()]
        if data.dtype == object:
            return ''.join(chr(int(f[r][()])) for r in data.ravel()).strip()
        return ''.join(chr(c) for c in data.ravel()).strip()
    except Exception:
        return 'NaN'

def _load_cell_types_CB(filepath):
    cell_types = []
    with h5py.File(filepath, 'r') as f:
        eg    = f['eg_neurons']
        n_neu = eg['st_init'].shape[1]
        ct_ds = eg['cell_type']
        for n in range(n_neu):
            ct = _read_hdf5_str(f[ct_ds[0, n]], f)
            if not ct or ct.lower() in ('nan', ''):
                ct = 'NaN'
            cell_types.append(ct)
    return cell_types, n_neu


def _build_spk_count(st_list, n_trials, n_neu):
    """Build (n_bins, n_trials, n_neu) spike count array."""
    sc = np.zeros((n_bins, n_trials, n_neu), dtype=np.float32)
    for n_idx in range(n_neu):
        for tt in range(n_trials):
            spk = np.atleast_1d(st_list[n_idx][tt]).ravel()
            for i, (bs, be) in enumerate(bin_edges):
                sc[i, tt, n_idx] = np.sum((spk > bs) & (spk < be))
    return sc


def _safe_decode(s1, s2):
    if s1.shape[0] < MIN_TRIALS or s2.shape[0] < MIN_TRIALS:
        return np.nan
    return kalman_decode(s1, s2,
                         ratio_train_val=ratio_train_val,
                         ncv=ncv, n_jobs=1)


# ===========================================================================
# Worker: one bin, all contrasts, Kalman only
# ===========================================================================

def _process_bin_kalman(b, sc_init, sc_reach,
                        i_pp, i_DCnD, r_pp, r_hit, r_DCnD,
                        neu_mask=None):
    """Return dict of BAC values for bin b, Kalman only."""
    si = sc_init[b]
    sr = sc_reach[b]

    if neu_mask is not None:
        si = si[:, neu_mask]
        sr = sr[:, neu_mask]

    hit = r_hit == 1
    out = {}

    pairs_init = {
        'pp': (si[i_pp == 0],   si[i_pp == 1]),
        'D':  (si[i_DCnD == 1], si[(i_DCnD==2)|(i_DCnD==3)]),
        'C':  (si[i_DCnD == 2], si[(i_DCnD==1)|(i_DCnD==3)]),
        'nD': (si[i_DCnD == 3], si[(i_DCnD==1)|(i_DCnD==2)]),
    }
    pairs_reach = {
        'pp': (sr[(r_pp==0)&hit],   sr[(r_pp==1)&hit]),
        'D':  (sr[(r_DCnD==1)&hit], sr[((r_DCnD==2)|(r_DCnD==3))&hit]),
        'C':  (sr[(r_DCnD==2)&hit], sr[((r_DCnD==1)|(r_DCnD==3))&hit]),
        'nD': (sr[(r_DCnD==3)&hit], sr[((r_DCnD==1)|(r_DCnD==2))&hit]),
    }

    for contrast in CONTRASTS:
        s1i, s2i = pairs_init[contrast]
        s1r, s2r = pairs_reach[contrast]
        out[f'{contrast}_init']  = _safe_decode(s1i, s2i)
        out[f'{contrast}_reach'] = _safe_decode(s1r, s2r)

    return out


def _results_to_bac(bin_results, prefix=''):
    """Convert list of per-bin dicts to {contrast: (n_bins, 2)} arrays."""
    bac = {}
    for contrast in CONTRASTS:
        ki = f'{prefix}{contrast}_init'
        kr = f'{prefix}{contrast}_reach'
        bac[contrast] = np.array([
            [r.get(ki, np.nan), r.get(kr, np.nan)]
            for r in bin_results
        ])
    return bac


# ===========================================================================
# Process one region
# ===========================================================================

def add_kalman_to_sessions(sessions, region):
    updated = []
    for sess in sessions:
        mouse  = sess['mouse']
        s_name = sess['sess']

        # Find animal index
        m = next((i for i, a in enumerate(animals) if f'{i+1}_{a}' == mouse), None)
        if m is None:
            print(f'  {mouse} not found in animals list — skipping')
            updated.append(sess)
            continue

        sess_base  = os.path.join(base_mat, mouse, s_name)
        ephys_path = os.path.join(sess_base, f'eg_neurons_{region}.mat')
        behav_path = os.path.join(sess_base, 'behavior_fundamentals.mat')

        if not os.path.isfile(ephys_path) or not os.path.isfile(behav_path):
            print(f'  {mouse} {s_name}: files missing — skipping')
            updated.append(sess)
            continue

        print(f'  [{region}] {mouse} {s_name}', end='', flush=True)
        t0 = time.time()

        # ── Behavior ───────────────────────────────────────────────────────
        bhv    = _load_behavior(behav_path)
        inv_pp = _arr(bhv['init_invPush_invPull_idx']).astype(int)
        i_pp   = np.where(inv_pp == 1, 0, np.where(inv_pp == 2, 1, 3))
        i_DCnD = _arr(bhv['DCnD_init']).astype(int)
        r_pp   = _arr(bhv['pp_inVec_idx']).astype(int)
        r_DCnD = _arr(bhv['DCnD_inVec_idx']).astype(int)
        r_hit  = np.nan_to_num(
            np.atleast_1d(bhv['hit_inVec']).ravel().astype(float), nan=0.0
        ).astype(int)

        n_trials_pp = len(i_pp)
        n_trials_rr = len(r_hit)

        # ── Spike counts ───────────────────────────────────────────────────
        n_neu = _n_neu(ephys_path)
        st_init, st_reach = _load_spike_times_BG(ephys_path, n_neu)
        sc_init  = _build_spk_count(st_init,  n_trials_pp, n_neu)
        sc_reach = _build_spk_count(st_reach, n_trials_rr, n_neu)

        # ── Kalman decoder ─────────────────────────────────────────────────
        if region == 'BG':
            bin_results = Parallel(n_jobs=N_JOBS_BINS, prefer='threads')(
                delayed(_process_bin_kalman)(
                    b, sc_init, sc_reach,
                    i_pp, i_DCnD, r_pp, r_hit, r_DCnD
                ) for b in range(n_bins)
            )
            kalman_bac = _results_to_bac(bin_results)

            # Insert into session dict
            sess = dict(sess)
            bac  = dict(sess.get('bac', {}))
            bac['kalman'] = kalman_bac
            sess['bac']   = bac

        else:  # CB — one mask per cell type
            cell_types, _ = _load_cell_types_CB(ephys_path)
            ct_arr        = np.array(cell_types)

            sess = dict(sess)
            bac  = dict(sess.get('bac', {}))

            if 'kalman' not in bac:
                bac['kalman'] = {}

            for ct_label in CB_CELL_TYPES:
                mask = (np.ones(n_neu, dtype=bool) if ct_label == 'all'
                        else ct_arr == ct_label)
                if mask.sum() == 0:
                    continue

                bin_results = Parallel(n_jobs=N_JOBS_BINS, prefer='threads')(
                    delayed(_process_bin_kalman)(
                        b, sc_init, sc_reach,
                        i_pp, i_DCnD, r_pp, r_hit, r_DCnD,
                        neu_mask=mask
                    ) for b in range(n_bins)
                )
                bac['kalman'][ct_label] = _results_to_bac(bin_results)

            sess['bac'] = bac

        print(f'  {time.time()-t0:.0f}s')
        updated.append(sess)

    return updated


# ===========================================================================
# MAIN
# ===========================================================================

print('=== Adding Kalman filter to BG npz ===')
bg_sessions = np.load(BG_NPZ, allow_pickle=True)['SVM_sess'].tolist()
print(f'Loaded {len(bg_sessions)} BG sessions')

bg_updated = add_kalman_to_sessions(bg_sessions, 'BG')
np.savez(BG_NPZ, SVM_sess=np.array(bg_updated, dtype=object),
         allow_pickle=True)
print(f'Saved updated BG npz → {BG_NPZ}')

print('\n=== Adding Kalman filter to CB npz ===')
cb_sessions = np.load(CB_NPZ, allow_pickle=True)['SVM_sess'].tolist()
print(f'Loaded {len(cb_sessions)} CB sessions')

cb_updated = add_kalman_to_sessions(cb_sessions, 'CB')
np.savez(CB_NPZ, SVM_sess=np.array(cb_updated, dtype=object),
         allow_pickle=True)
print(f'Saved updated CB npz → {CB_NPZ}')

## 4. Full comparison — `compare_all.py` (robust version)
Comprehensive analysis: per-session / per-animal / grand-average decoding, BG vs CB, AUC, pointwise (Wilcoxon + Cohen's d), latency, and per-animal statistics for all four decoders. Handles the CB Kalman nesting via `_lookup_bac`.

In [ ]:
"""
compare_all.py  — versão robusta
=================================
Análise completa de decoding para BG, CB e BG+CB.

Novidades em relação à versão anterior:
  ★ AUC total e em janelas de 0.5 s (qual época é mais informativa)
  ★ Stats por animal (Wilcoxon entre sessões dentro do animal)
  ★ Plot per-animal BG vs CB com SEM entre sessões (decoder à escolha)
  ★ Análise pontual (pointwise): teste estatístico em cada bin temporal
      → quando é que BG e CB diferem significativamente?
      → quando é que trocam de liderança?
  ★ Effect size (Cohen's d) BG vs CB por bin
  ★ Latência ao pico e tempo para acima do chance
  ★ Correlação entre curvas BG e CB por animal

Outputs (SAVE_DIR/):
  per_session/{region}/{animal}_{sess}.png
  per_animal/{region}/{animal}.png
  per_animal/BG_vs_CB/{animal}_BG_vs_CB_{contrast}.png   ← novo
  grand/{region}/grand_{contrast}.png
  grand/BG_vs_CB_{contrast}.png
  auc/{region}_auc_{contrast}_{align}.csv                 ← novo (AUC)
  auc/auc_windows_{contrast}_{align}.png                  ← novo
  pointwise/pointwise_{contrast}_{align}.png              ← novo
  stats/decoder_stats_BG.csv
  stats/decoder_stats_CB.csv
  stats/BG_vs_CB_stats.csv
  stats/per_animal_stats.csv                              ← novo
  stats/latency_stats.csv                                 ← novo
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from itertools import combinations
from scipy import stats
from scipy.integrate import trapezoid

# ===========================================================================
# Configuration
# ===========================================================================
BG_NPZ   = r'data/SVM_sess_BG.npz'
CB_NPZ   = r'data/SVM_sess_CB.npz'
SAVE_DIR = r'output/comparison_full7'

CB_CELL_TYPE   = 'all'
DECODER_PLOT   = 'svm'   # decoder usado nos plots per-animal BG vs CB
AUC_WINDOW     = 0.5     # tamanho da janela AUC em segundos
ALPHA          = 0.05    # threshold de significância (bonferroni aplicado)
MIN_RELIABLE_SESSIONS = 3   # abaixo disto, animal marcado como pouco fiável
MAJORITY_THRESH       = 0.75  # fracção de sessões para sombrear consistência
CHANCE         = 0.5

METHODS       = ['svm', 'lda', 'nb', 'kalman']
METHOD_LABELS = {'svm':'SVM','lda':'LDA','nb':'Naive Bayes','kalman':'Kalman'}
METHOD_COLORS = {'svm':'#2E75B6','lda':'#70AD47','nb':'#ED7D31','kalman':'#7030A0'}
METHOD_LS     = {'svm':'-','lda':'--','nb':':','kalman':'-.'}

CONTRASTS       = ['pp','D','C','nD']
CONTRAST_LABELS = {'pp':'Push / Pull','D':'Dominant',
                   'C':'Center','nD':'Non-dominant'}
CONTRAST_COLORS = {'pp':'#1F4E79','D':'#2E75B6',
                   'C':'#70AD47','nD':'#ED7D31'}
ALIGN_LABELS    = {0:'Init',1:'Reach'}

REGION_COLORS = {'BG':'#1F4E79','CB':'#C00000'}
REGION_LS     = {'BG':'-','CB':'--'}

EXCLUSIONS = [
    ('3_Toblerone','R2','both'),('3_Toblerone','R3','both'),
    ('3_Toblerone','R4','both'),('3_Toblerone','R6','both'),
    ('3_Milka','R7','both'),
]

for sub in [
    'per_session/BG','per_session/CB',
    'per_animal/BG','per_animal/CB','per_animal/BG_vs_CB',
    'grand/BG','grand/CB','grand',
    'auc','pointwise','stats',
]:
    os.makedirs(os.path.join(SAVE_DIR,sub), exist_ok=True)


# ===========================================================================
# Loaders
# ===========================================================================

def load_sessions(path):
    return np.load(path,allow_pickle=True)['SVM_sess'].tolist()

def is_excluded(animal,sess,region):
    for ea,es,er in EXCLUSIONS:
        if animal==ea and sess==es and (er=='both' or er==region):
            return True
    return False

_MISSING_WARNED = set()

def _lookup_bac(bac, region, m, contrast, cb_ct):
    """
    Devolve o array (n_bins, 2) para um método, lidando com as duas
    estruturas possíveis:
      BG:  bac[m][contrast]
      CB:  bac[cb_ct][m][contrast]   (aninhado sob tipo celular)
    Fallback: se a estrutura CB aninhada não tiver o método (ex: Kalman
    adicionado no nível errado por add_kalman.py), tenta bac[m][contrast].
    """
    if region == 'BG':
        return np.array(bac[m][contrast])
    # ── CB ──────────────────────────────────────────────────────────────
    # Estrutura 1 (SVM/LDA/NB): bac[cb_ct][m][contrast]  (tipo celular -> método)
    try:
        return np.array(bac[cb_ct][m][contrast])
    except (KeyError, TypeError, IndexError):
        pass
    # Estrutura 2 (Kalman, add_kalman.py): bac[m][cb_ct][contrast]  (método -> tipo celular)
    try:
        return np.array(bac[m][cb_ct][contrast])
    except (KeyError, TypeError, IndexError):
        pass
    # fallback 1: método no nível de topo (estilo BG) dentro do npz CB
    try:
        return np.array(bac[m][contrast])
    except (KeyError, TypeError, IndexError):
        pass
    # fallback 2: método aninhado sob qualquer tipo celular (tipo -> método)
    if isinstance(bac, dict):
        for ct, sub in bac.items():
            if isinstance(sub, dict) and m in sub:
                try:
                    return np.array(sub[m][contrast])
                except (KeyError, TypeError, IndexError):
                    pass
    raise KeyError(f'{m}/{contrast} não encontrado (cb_ct={cb_ct})')

def get_bac(sess,region,contrast,align,cb_ct='all'):
    bac = sess.get('bac',{})
    result = {}
    for m in METHODS:
        try:
            arr = _lookup_bac(bac, region, m, contrast, cb_ct)
            result[m] = arr[:,align]
        except Exception:
            result[m] = None
            # avisa UMA vez por (região, método) que está em falta
            key = (region, m)
            if key not in _MISSING_WARNED:
                _MISSING_WARNED.add(key)
                print(f'  [AVISO] {m.upper()} em falta no npz de {region} '
                      f'(contraste {contrast}) — verifica se add_kalman.py '
                      f'correu para este ficheiro e no nível certo.')
    return result

def get_bin_info(sessions):
    for s in sessions:
        if 'bin_edges' in s:
            be = s['bin_edges']
            bw = be[0,1]-be[0,0]
            bm = be[:,0]+bw/2
            return bm,[bm[0],bm[-1]]
    return None,None


# ===========================================================================
# Data collection
# ===========================================================================

def collect_region(sessions,region,cb_ct='all'):
    bm,wi = get_bin_info(sessions)
    animals,by_animal = [],{}
    for sess in sessions:
        animal,s_name = sess['mouse'],sess['sess']
        if is_excluded(animal,s_name,region):
            print(f'  [EXCL] {region} {animal} {s_name}'); continue
        if animal not in animals: animals.append(animal)
        if animal not in by_animal:
            by_animal[animal] = {m:{c:{a:[] for a in [0,1]}
                                    for c in CONTRASTS} for m in METHODS}
        for c in CONTRASTS:
            for a in [0,1]:
                bac = get_bac(sess,region,c,a,cb_ct)
                for m in METHODS:
                    arr = bac.get(m)
                    if arr is not None and not np.all(np.isnan(arr)):
                        by_animal[animal][m][c][a].append(arr)
    return by_animal,animals,bm,wi


def animal_mean_sem(by_animal,animal,align):
    means,sems = {},{}
    for m in METHODS:
        means[m]={};sems[m]={}
        for c in CONTRASTS:
            arrs = by_animal[animal][m][c][align]
            if not arrs: continue
            s = np.array(arrs)
            means[m][c] = np.nanmean(s,axis=0)
            sems[m][c]  = np.nanstd(s,axis=0)/np.sqrt(s.shape[0])
    return means,sems


def grand_mean_sem(by_animal,animals,align):
    avgs = {m:{c:[] for c in CONTRASTS} for m in METHODS}
    for animal in animals:
        for m in METHODS:
            for c in CONTRASTS:
                arrs = by_animal[animal][m][c][align]
                if arrs:
                    avgs[m][c].append(np.nanmean(np.array(arrs),axis=0))
    gm,gs = {},{}
    for m in METHODS:
        gm[m]={};gs[m]={}
        for c in CONTRASTS:
            if not avgs[m][c]: continue
            s = np.array(avgs[m][c])
            gm[m][c] = np.nanmean(s,axis=0)
            gs[m][c] = np.nanstd(s,axis=0)/np.sqrt(s.shape[0])
    return gm,gs,len(animals)


# ===========================================================================
# AUC analysis
# ===========================================================================

def compute_auc(curve, bin_mean, window=0.5):
    """
    Returns:
      auc_total  : float — AUC over full window (trapezoidal)
      auc_windows: dict  — {(t_start, t_end): auc_value}
    """
    t0,t1 = bin_mean[0],bin_mean[-1]
    auc_total = float(trapezoid(curve,bin_mean))

    auc_windows = {}
    t = t0
    while t < t1-1e-9:
        te = min(t+window, t1)
        mask = (bin_mean>=t) & (bin_mean<=te)
        if mask.sum()>=2:
            auc_windows[(round(t,3),round(te,3))] = float(
                trapezoid(curve[mask],bin_mean[mask])
            )
        t = te
    return auc_total, auc_windows


def run_auc_analysis(by_animal,animals,region,bin_mean):
    """Per-animal AUC total and per window. Returns DataFrame."""
    rows = []
    for animal in animals:
        for m in METHODS:
            for c in CONTRASTS:
                for a in [0,1]:
                    arrs = by_animal[animal][m][c][a]
                    if not arrs: continue
                    curve = np.nanmean(np.array(arrs),axis=0)
                    auc_total,auc_wins = compute_auc(curve,bin_mean)
                    row = {'region':region,'animal':animal,
                           'decoder':METHOD_LABELS[m],
                           'contrast':CONTRAST_LABELS[c],
                           'alignment':ALIGN_LABELS[a],
                           'auc_total':round(auc_total,4)}
                    for (ts,te),val in auc_wins.items():
                        row[f'auc_{ts}_{te}s'] = round(val,4)
                    rows.append(row)
    return pd.DataFrame(rows)


def plot_auc_windows(bg_auc_df, cb_auc_df, bin_mean, contrast, align):
    """Barplot: AUC per 0.5s window, BG vs CB."""
    # Get window columns
    al = ALIGN_LABELS[align]
    cl = CONTRAST_LABELS[contrast]
    m  = METHOD_LABELS[DECODER_PLOT]

    win_cols = [c for c in bg_auc_df.columns if c.startswith('auc_') and 's' in c]
    if not win_cols: return

    fig,ax = plt.subplots(figsize=(14,5))
    x   = np.arange(len(win_cols))
    w   = 0.35

    for shift,df,region in [(-w/2,bg_auc_df,'BG'),(w/2,cb_auc_df,'CB')]:
        sub = df[(df['decoder']==m) &
                 (df['contrast']==cl) &
                 (df['alignment']==al)]
        if sub.empty: continue
        means = sub[win_cols].mean()
        sems  = sub[win_cols].sem()
        ax.bar(x+shift,means,w,yerr=sems,color=REGION_COLORS[region],
               alpha=0.8,label=region,capsize=4)

    # x tick labels
    labels = [c.replace('auc_','').replace('_',' → ') for c in win_cols]
    ax.set_xticks(x); ax.set_xticklabels(labels,rotation=45,ha='right',fontsize=8)
    ax.axhline(CHANCE*AUC_WINDOW,ls='--',color='grey',lw=1,alpha=0.5,
               label=f'chance ({CHANCE}×{AUC_WINDOW}s)')
    ax.set_ylabel('AUC (BAC × s)',fontsize=9)
    ax.set_title(f'AUC per {AUC_WINDOW}s window — {m} — {cl} — {al}',
                 fontsize=11,fontweight='bold')
    ax.legend(frameon=False,fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    fname = os.path.join(SAVE_DIR,'auc',f'auc_windows_{contrast}_{al}.png')
    fig.savefig(fname,dpi=150); plt.close(fig)
    print(f'  saved auc_windows_{contrast}_{al}.png')


# ===========================================================================
# Pointwise analysis — when do BG and CB differ?
# ===========================================================================

def pointwise_analysis(bg_by_animal, bg_animals,
                       cb_by_animal, cb_animals,
                       bin_mean, contrast, align):
    """
    At each time bin:
      1. Compute BG and CB BAC per animal (mean of sessions)
      2. Wilcoxon signed-rank (paired on common animals)
      3. Compute Cohen's d effect size
      4. Find where CB > BG, BG > CB, and crossover points
    Returns: dict with p-values, d, difference, significant mask
    """
    common = sorted(set(bg_animals) & set(cb_animals))
    n_bins = len(bin_mean)
    m      = DECODER_PLOT

    bg_curves = []
    cb_curves = []
    for animal in common:
        bg_arrs = bg_by_animal[animal][m][contrast][align]
        cb_arrs = cb_by_animal[animal][m][contrast][align]
        if not bg_arrs or not cb_arrs: continue
        bg_curves.append(np.nanmean(np.array(bg_arrs),axis=0))
        cb_curves.append(np.nanmean(np.array(cb_arrs),axis=0))

    if len(bg_curves)<3:
        return None

    BG = np.array(bg_curves)  # (n_animals, n_bins)
    CB = np.array(cb_curves)

    p_vals = np.full(n_bins, np.nan)
    d_vals = np.full(n_bins, np.nan)

    for b in range(n_bins):
        bg_b = BG[:,b]; cb_b = CB[:,b]
        valid = ~(np.isnan(bg_b)|np.isnan(cb_b))
        if valid.sum()<3: continue
        try:
            _,p = stats.wilcoxon(bg_b[valid],cb_b[valid],alternative='two-sided')
            p_vals[b] = p
        except: pass
        # Cohen's d on difference
        diff = (cb_b-bg_b)[valid]
        if diff.std()>0:
            d_vals[b] = diff.mean()/diff.std()

    # Bonferroni correction
    n_tests  = np.sum(~np.isnan(p_vals))
    p_corr   = np.where(np.isnan(p_vals), np.nan, np.minimum(p_vals*n_tests,1.0))
    sig_mask = p_corr < ALPHA

    # Mean difference CB - BG
    diff_mean = np.nanmean(CB-BG, axis=0)

    # Crossover points (where sign changes)
    crossovers = []
    for b in range(1,n_bins):
        if np.sign(diff_mean[b])!=np.sign(diff_mean[b-1]):
            crossovers.append(bin_mean[b])

    return {'p_vals':p_vals,'p_corr':p_corr,'sig_mask':sig_mask,
            'diff_mean':diff_mean,'d_vals':d_vals,
            'crossovers':crossovers,'n_animals':len(bg_curves)}


def plot_pointwise(bg_by_animal,bg_animals,cb_by_animal,cb_animals,
                   bin_mean,win_int,contrast,align):
    res = pointwise_analysis(bg_by_animal,bg_animals,cb_by_animal,cb_animals,
                             bin_mean,contrast,align)
    if res is None:
        print(f'  pointwise {contrast} {ALIGN_LABELS[align]}: not enough data'); return

    m  = DECODER_PLOT
    cl = CONTRAST_LABELS[contrast]
    al = ALIGN_LABELS[align]

    # Get grand mean curves for plotting
    bg_gm,bg_gs,_ = grand_mean_sem(bg_by_animal,bg_animals,align)
    cb_gm,cb_gs,_ = grand_mean_sem(cb_by_animal,cb_animals,align)
    bg_c = bg_gm.get(m,{}).get(contrast)
    cb_c = cb_gm.get(m,{}).get(contrast)
    bg_s = bg_gs.get(m,{}).get(contrast)
    cb_s = cb_gs.get(m,{}).get(contrast)

    fig,(ax1,ax2,ax3) = plt.subplots(3,1,figsize=(14,12),sharex=True)
    fig.suptitle(f'Pointwise analysis — {METHOD_LABELS[m]} — {cl} — {al}'
                 f'  (n={res["n_animals"]} animals, Bonferroni corrected)',
                 fontsize=11,fontweight='bold')

    # ── Panel 1: BAC curves + significant regions ──────────────────────────
    for arr,sem,region in [(bg_c,bg_s,'BG'),(cb_c,cb_s,'CB')]:
        if arr is None: continue
        ax1.plot(bin_mean,arr,color=REGION_COLORS[region],
                 lw=2,ls=REGION_LS[region],label=region)
        if sem is not None:
            ax1.fill_between(bin_mean,arr-sem,arr+sem,
                             color=REGION_COLORS[region],alpha=0.12)

    # Shade significant regions
    sig = res['sig_mask']
    if sig.any():
        ax1.fill_between(bin_mean,CHANCE,1.0,where=sig,
                         color='gold',alpha=0.25,label=f'p<{ALPHA} (Bonferroni)')

    # Mark crossovers
    for xo in res['crossovers']:
        ax1.axvline(xo,color='purple',lw=1.5,ls=':',alpha=0.7)

    ax1.axhline(CHANCE,ls='--',color='grey',lw=1,alpha=0.5)
    ax1.axvline(0,ls='-.',color='grey',lw=1,alpha=0.3)
    ax1.set_xlim(win_int); ax1.set_ylim([0.44,1.0])
    ax1.set_ylabel('BAC',fontsize=9)
    ax1.legend(frameon=False,fontsize=8,loc='upper left')
    ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)
    ax1.set_title('BAC curves (shaded = significant difference CB≠BG)')

    # ── Panel 2: Difference CB - BG + effect size ─────────────────────────
    diff = res['diff_mean']
    ax2.plot(bin_mean,diff,color='black',lw=2,label='CB − BG')
    ax2.fill_between(bin_mean,0,diff,where=diff>0,color=REGION_COLORS['CB'],
                     alpha=0.3,label='CB > BG')
    ax2.fill_between(bin_mean,0,diff,where=diff<0,color=REGION_COLORS['BG'],
                     alpha=0.3,label='BG > CB')

    # Mark significant bins with dots
    sig_bins = bin_mean[res['sig_mask']]
    sig_diff = diff[res['sig_mask']]
    ax2.scatter(sig_bins,sig_diff,color='gold',s=20,zorder=5,label='significant')

    ax2.axhline(0,color='grey',lw=1,ls='--')
    ax2.axvline(0,ls='-.',color='grey',lw=1,alpha=0.3)
    ax2.set_ylabel('ΔBAC (CB − BG)',fontsize=9)
    ax2.legend(frameon=False,fontsize=8)
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
    ax2.set_title("Difference CB − BG  (gold dots = Bonferroni significant)")

    # Mark crossovers with text
    for xo in res['crossovers']:
        ax2.axvline(xo,color='purple',lw=1.5,ls=':',alpha=0.7)
        ax2.text(xo,ax2.get_ylim()[0]*0.95,f'{xo:.2f}s',
                 fontsize=7,color='purple',ha='center')

    # ── Panel 3: Cohen's d ────────────────────────────────────────────────
    d = res['d_vals']
    ax3.plot(bin_mean,d,color='darkgreen',lw=2)
    ax3.fill_between(bin_mean,0,d,where=d>0,color='lightgreen',alpha=0.4)
    ax3.fill_between(bin_mean,0,d,where=d<0,color='lightcoral',alpha=0.4)
    ax3.axhline(0,color='grey',lw=1,ls='--')
    ax3.axhline(0.5,color='grey',lw=0.8,ls=':',alpha=0.5,label="d=0.5 (medium)")
    ax3.axhline(-0.5,color='grey',lw=0.8,ls=':',alpha=0.5)
    ax3.axvline(0,ls='-.',color='grey',lw=1,alpha=0.3)
    ax3.set_xlim(win_int)
    ax3.set_xlabel(f'time from {"init" if align==0 else "reach"} (s)',fontsize=9)
    ax3.set_ylabel("Cohen's d",fontsize=9)
    ax3.legend(frameon=False,fontsize=8)
    ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)
    ax3.set_title("Effect size Cohen's d  (CB − BG, positive = CB stronger)")

    plt.tight_layout()
    fname = os.path.join(SAVE_DIR,'pointwise',
                         f'pointwise_{contrast}_{al}.png')
    fig.savefig(fname,dpi=150); plt.close(fig)
    print(f'  saved pointwise_{contrast}_{al}.png')

    return res


# ===========================================================================
# Per-animal BG vs CB plot
# ===========================================================================

def plot_animal_bg_cb(bg_by_animal, cb_by_animal, animal,
                      bin_mean, win_int, contrast):
    """One animal, one contrast, both regions, both alignments — with sessions."""
    fig,axes = plt.subplots(1,2,figsize=(13,4.5),sharey=True)
    fig.suptitle(f'{animal} — BG vs CB — {METHOD_LABELS[DECODER_PLOT]}'
                 f' — {CONTRAST_LABELS[contrast]}',
                 fontsize=11,fontweight='bold')

    for align,ax in zip([0,1],axes):
        for region,by_animal in [('BG',bg_by_animal),('CB',cb_by_animal)]:
            if animal not in by_animal: continue
            arrs = by_animal[animal][DECODER_PLOT][contrast][align]
            if not arrs: continue
            s    = np.array(arrs)
            mean = np.nanmean(s,axis=0)
            sem  = np.nanstd(s,axis=0)/np.sqrt(s.shape[0])
            ax.plot(bin_mean,mean,color=REGION_COLORS[region],
                    lw=2,ls=REGION_LS[region],label=f'{region} (n={len(arrs)} sess)')
            ax.fill_between(bin_mean,mean-sem,mean+sem,
                            color=REGION_COLORS[region],alpha=0.15)
            for arr in arrs:
                ax.plot(bin_mean,arr,color=REGION_COLORS[region],lw=0.5,alpha=0.25)
        ax.axhline(CHANCE,ls='--',color='grey',lw=1,alpha=0.5)
        ax.axvline(0,ls='-.',color='grey',lw=1,alpha=0.3)
        ax.set_xlim(win_int); ax.set_ylim([0.44,1.0])
        ax.set_xlabel(f'time from {"init" if align==0 else "reach"} (s)',fontsize=9)
        ax.set_ylabel('BAC',fontsize=9)
        ax.set_title(ALIGN_LABELS[align],fontsize=10)
        ax.legend(frameon=False,fontsize=8)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    fname = os.path.join(SAVE_DIR,'per_animal','BG_vs_CB',
                         f'{animal}_BG_vs_CB_{contrast}.png')
    fig.savefig(fname,dpi=150); plt.close(fig)


def plot_animal_bg_cb_all_contrasts(bg_by_animal, cb_by_animal, animal,
                                    bin_mean, win_int):
    """
    One animal — all 4 contrasts in one figure.
    4 rows (contrasts) × 2 cols (Init / Reach).
    BG solid, CB dashed. Mean ± SEM across sessions + individual sessions faint.
    """
    n_bg = len(bg_by_animal.get(animal,{}).get(DECODER_PLOT,{})
               .get('pp',{}).get(0,[]))
    n_cb = len(cb_by_animal.get(animal,{}).get(DECODER_PLOT,{})
               .get('pp',{}).get(0,[]))
    unreliable = min(n_bg, n_cb) < MIN_RELIABLE_SESSIONS

    fig,axes = plt.subplots(len(CONTRASTS),2,
                             figsize=(13, 4.5*len(CONTRASTS)),
                             sharey=False)
    title = (f'{animal} — BG vs CB — {METHOD_LABELS[DECODER_PLOT]}'
             f'  (BG n={n_bg}, CB n={n_cb} sessions)'
             f'\nAll contrasts (mean ± SEM across sessions)')
    if unreliable:
        title += (f'\n⚠ UNRELIABLE: fewer than {MIN_RELIABLE_SESSIONS} '
                  f'usable sessions — interpret with extreme caution')
    fig.suptitle(title, fontsize=12, fontweight='bold',
                 color=('#C00000' if unreliable else 'black'))

    for row,contrast in enumerate(CONTRASTS):
        for col,align in enumerate([0,1]):
            ax = axes[row,col]
            for region,by_animal in [('BG',bg_by_animal),('CB',cb_by_animal)]:
                if animal not in by_animal: continue
                arrs = by_animal[animal][DECODER_PLOT][contrast][align]
                if not arrs: continue
                s    = np.array(arrs)
                mean = np.nanmean(s,axis=0)
                sem  = np.nanstd(s,axis=0)/np.sqrt(s.shape[0])
                ax.plot(bin_mean,mean,color=REGION_COLORS[region],
                        lw=2,ls=REGION_LS[region],
                        label=f'{region} (n={len(arrs)})')
                ax.fill_between(bin_mean,mean-sem,mean+sem,
                                color=REGION_COLORS[region],alpha=0.15)
                for arr in arrs:
                    ax.plot(bin_mean,arr,color=REGION_COLORS[region],
                            lw=0.4,alpha=0.2)
            ax.axhline(CHANCE,ls='--',color='grey',lw=1,alpha=0.5)
            ax.axvline(0,ls='-.',color='grey',lw=1,alpha=0.3)
            ax.set_xlim(win_int); ax.set_ylim([0.44,1.0])
            ax.set_xlabel(f'time from {"init" if align==0 else "reach"} (s)',fontsize=8)
            ax.set_ylabel('BAC',fontsize=8)
            ax.set_title(f'{CONTRAST_LABELS[contrast]} — {ALIGN_LABELS[align]}',
                         fontsize=9)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            if row==0 and col==1:
                ax.legend(frameon=False,fontsize=8)

    plt.tight_layout()
    fname = os.path.join(SAVE_DIR,'per_animal','BG_vs_CB',
                         f'{animal}_BG_vs_CB_all_contrasts.png')
    fig.savefig(fname,dpi=150); plt.close(fig)
    print(f'    saved {animal}_BG_vs_CB_all_contrasts.png')


def plot_animal_pointwise(bg_by_animal, cb_by_animal, animal,
                          bin_mean, win_int):
    """
    Per-animal pointwise: difference CB−BG for each contrast × alignment.
    Uses individual sessions within the animal (sign test across sessions).
    4 rows (contrasts) × 2 cols (Init / Reach).
    """
    # Determine n sessions (use Push/Pull Init as reference)
    n_bg = len(bg_by_animal.get(animal,{}).get(DECODER_PLOT,{})
               .get('pp',{}).get(0,[]))
    n_cb = len(cb_by_animal.get(animal,{}).get(DECODER_PLOT,{})
               .get('pp',{}).get(0,[]))
    n_min = min(n_bg, n_cb)
    unreliable = n_min < MIN_RELIABLE_SESSIONS

    fig,axes = plt.subplots(len(CONTRASTS),2,
                             figsize=(13, 3.5*len(CONTRASTS)),
                             sharey=False)
    title = (f'{animal} — Pointwise CB − BG — {METHOD_LABELS[DECODER_PLOT]}'
             f'  (BG n={n_bg}, CB n={n_cb} sessions)'
             f'\n(shaded = CB > BG in ≥{int(MAJORITY_THRESH*100)}% of sessions)')
    if unreliable:
        title += (f'\n⚠ UNRELIABLE: fewer than {MIN_RELIABLE_SESSIONS} '
                  f'usable sessions — interpret with extreme caution')
    fig.suptitle(title, fontsize=12, fontweight='bold',
                 color=('#C00000' if unreliable else 'black'))

    for row,contrast in enumerate(CONTRASTS):
        for col,align in enumerate([0,1]):
            ax = axes[row,col]

            bg_arrs = (bg_by_animal.get(animal,{})
                       .get(DECODER_PLOT,{}).get(contrast,{}).get(align,[]))
            cb_arrs = (cb_by_animal.get(animal,{})
                       .get(DECODER_PLOT,{}).get(contrast,{}).get(align,[]))

            if not bg_arrs or not cb_arrs:
                ax.set_visible(False); continue

            bg_mean = np.nanmean(np.array(bg_arrs),axis=0)
            cb_mean = np.nanmean(np.array(cb_arrs),axis=0)
            diff    = cb_mean - bg_mean

            # Session-level differences for consistency ribbon
            n_sess = min(len(bg_arrs),len(cb_arrs))
            sess_diffs = [np.array(cb_arrs[i]) - np.array(bg_arrs[i])
                          for i in range(n_sess)]

            # Fraction of sessions where CB > BG at each bin
            if sess_diffs:
                frac_cb_wins = np.mean(
                    [d > 0 for d in sess_diffs], axis=0
                )
            else:
                frac_cb_wins = np.full(len(bin_mean), np.nan)

            # Plot individual session differences faintly
            for sd in sess_diffs:
                ax.plot(bin_mean,sd,color='grey',lw=0.5,alpha=0.2)

            # Mean difference
            ax.plot(bin_mean,diff,color='black',lw=2,label='mean diff')
            ax.fill_between(bin_mean,0,diff,where=diff>0,
                            color=REGION_COLORS['CB'],alpha=0.3,label='CB > BG')
            ax.fill_between(bin_mean,0,diff,where=diff<0,
                            color=REGION_COLORS['BG'],alpha=0.3,label='BG > CB')

            # Shade where majority of sessions agree (only if reliable)
            if not unreliable:
                majority = frac_cb_wins > MAJORITY_THRESH
                ax.fill_between(bin_mean,
                                np.full(len(bin_mean),-0.5),
                                np.full(len(bin_mean), 0.5),
                                where=majority,
                                color=REGION_COLORS['CB'],alpha=0.08)

            ax.axhline(0,color='grey',lw=1,ls='--')
            ax.axvline(0,ls='-.',color='grey',lw=1,alpha=0.3)
            ax.set_xlim(win_int)
            ax.set_xlabel(f'time from {"init" if align==0 else "reach"} (s)',
                          fontsize=8)
            ax.set_ylabel('ΔBAC (CB−BG)',fontsize=8)
            ax.set_title(f'{CONTRAST_LABELS[contrast]} — {ALIGN_LABELS[align]}',
                         fontsize=9)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            if row==0 and col==1:
                ax.legend(frameon=False,fontsize=7)

    plt.tight_layout()
    fname = os.path.join(SAVE_DIR,'per_animal','BG_vs_CB',
                         f'{animal}_pointwise_all_contrasts.png')
    fig.savefig(fname,dpi=150); plt.close(fig)
    print(f'    saved {animal}_pointwise_all_contrasts.png')


# ===========================================================================
# Latency analysis
# ===========================================================================

def compute_latency_stats(by_animal,animals,region,bin_mean,threshold=0.55):
    """
    For each animal × decoder × contrast × align:
      - Latency to peak (time of max BAC)
      - Latency to above-chance (first bin > threshold, sustained 200ms)
    """
    rows = []
    sustain = max(1,int(0.2/(bin_mean[1]-bin_mean[0])))  # bins for 200ms
    for animal in animals:
        for m in METHODS:
            for c in CONTRASTS:
                for a in [0,1]:
                    arrs = by_animal[animal][m][c][a]
                    if not arrs: continue
                    curve = np.nanmean(np.array(arrs),axis=0)
                    # Peak latency
                    peak_idx = np.nanargmax(curve)
                    peak_t   = bin_mean[peak_idx]
                    peak_bac = curve[peak_idx]
                    # Time to above chance (sustained)
                    above = curve > threshold
                    lat_t = np.nan
                    for b in range(len(above)-sustain):
                        if all(above[b:b+sustain]):
                            lat_t = bin_mean[b]; break
                    rows.append({
                        'region':region,'animal':animal,
                        'decoder':METHOD_LABELS[m],
                        'contrast':CONTRAST_LABELS[c],
                        'alignment':ALIGN_LABELS[a],
                        'peak_BAC':round(float(peak_bac),4),
                        'peak_time':round(float(peak_t),4),
                        'latency_above_chance':round(float(lat_t),4) if not np.isnan(lat_t) else np.nan,
                    })
    return pd.DataFrame(rows)


# ===========================================================================
# Stats functions (from before + new per-animal)
# ===========================================================================

def _peak(by_animal,animal,m,c,a):
    arrs = by_animal[animal][m][c][a]
    if not arrs: return np.nan
    return float(np.nanmax(np.nanmean(np.array(arrs),axis=0)))

def _wilcoxon(a,b):
    valid = ~(np.isnan(a)|np.isnan(b))
    if valid.sum()<3: return np.nan,np.nan
    try:    return stats.wilcoxon(a[valid],b[valid],alternative='two-sided')
    except: return np.nan,np.nan

def compute_decoder_stats(by_animal,animals,region):
    rows = []
    for c in CONTRASTS:
        for a in [0,1]:
            peaks = {m:np.array([_peak(by_animal,an,m,c,a) for an in animals])
                     for m in METHODS}
            for m1,m2 in combinations(METHODS,2):
                stat,pval = _wilcoxon(peaks[m1],peaks[m2])
                mn1,mn2 = np.nanmean(peaks[m1]),np.nanmean(peaks[m2])
                rows.append({'region':region,'contrast':CONTRAST_LABELS[c],
                             'alignment':ALIGN_LABELS[a],
                             'decoder_1':METHOD_LABELS[m1],
                             'decoder_2':METHOD_LABELS[m2],
                             'peak_BAC_1':round(mn1,4),'peak_BAC_2':round(mn2,4),
                             'winner':METHOD_LABELS[m1] if mn1>=mn2 else METHOD_LABELS[m2],
                             'W_stat':round(stat,3) if not np.isnan(stat) else np.nan,
                             'p_value':round(pval,4) if not np.isnan(pval) else np.nan,
                             'significant':bool(pval<0.05) if not np.isnan(pval) else False,
                             'n_animals':int((~np.isnan(peaks[m1])&~np.isnan(peaks[m2])).sum())})
    return pd.DataFrame(rows)

def compute_region_stats(bg_by_animal,bg_animals,cb_by_animal,cb_animals):
    common = sorted(set(bg_animals)&set(cb_animals))
    rows   = []
    for m in METHODS:
        for c in CONTRASTS:
            for a in [0,1]:
                bg_p = np.array([_peak(bg_by_animal,an,m,c,a) for an in common])
                cb_p = np.array([_peak(cb_by_animal,an,m,c,a) for an in common])
                stat,pval = _wilcoxon(bg_p,cb_p)
                mbg = np.nanmean([_peak(bg_by_animal,an,m,c,a) for an in bg_animals])
                mcb = np.nanmean([_peak(cb_by_animal,an,m,c,a) for an in cb_animals])
                rows.append({'decoder':METHOD_LABELS[m],'contrast':CONTRAST_LABELS[c],
                             'alignment':ALIGN_LABELS[a],
                             'peak_BAC_BG':round(mbg,4),'peak_BAC_CB':round(mcb,4),
                             'winner':'BG' if mbg>=mcb else 'CB',
                             'test':'wilcoxon_paired','W_stat':round(stat,3) if not np.isnan(stat) else np.nan,
                             'p_value':round(pval,4) if not np.isnan(pval) else np.nan,
                             'significant':bool(pval<0.05) if not np.isnan(pval) else False,
                             'n':int((~np.isnan(bg_p)&~np.isnan(cb_p)).sum())})
    return pd.DataFrame(rows)

def compute_per_animal_stats(by_animal,animals,region):
    """Wilcoxon within each animal across sessions."""
    rows = []
    for animal in animals:
        for m in METHODS:
            for c in CONTRASTS:
                for a in [0,1]:
                    arrs = by_animal[animal][m][c][a]
                    if len(arrs)<3: continue
                    peaks = [float(np.nanmax(arr)) for arr in arrs]
                    # Test: is mean peak > chance?
                    try:
                        stat,pval = stats.wilcoxon(
                            [p-CHANCE for p in peaks],
                            alternative='greater'
                        )
                    except:
                        stat,pval = np.nan,np.nan
                    rows.append({'region':region,'animal':animal,
                                 'decoder':METHOD_LABELS[m],
                                 'contrast':CONTRAST_LABELS[c],
                                 'alignment':ALIGN_LABELS[a],
                                 'n_sessions':len(arrs),
                                 'mean_peak_BAC':round(np.nanmean(peaks),4),
                                 'sem_peak_BAC':round(np.nanstd(peaks)/np.sqrt(len(peaks)),4),
                                 'W_stat':round(stat,3) if not np.isnan(stat) else np.nan,
                                 'p_value':round(pval,4) if not np.isnan(pval) else np.nan,
                                 'above_chance':bool(pval<0.05) if not np.isnan(pval) else False})
    return pd.DataFrame(rows)

def print_summary(df_bg,df_cb,df_vs):
    print('\n'+'='*65)
    print('WILCOXON RESULTS SUMMARY')
    print('='*65)
    for label,df in [('BG — melhor decoder',df_bg),('CB — melhor decoder',df_cb)]:
        sig = df[df['significant']]
        print(f'\n{label}  (p<0.05: {len(sig)}/{len(df)} pares):')
        if len(sig)==0: print('  Nenhuma diferença significativa.')
        else:
            for _,r in sig.iterrows():
                print(f"  {r['contrast']:14s} {r['alignment']:6s}  "
                      f"{r['decoder_1']} vs {r['decoder_2']}  →  {r['winner']}"
                      f"  (p={r['p_value']:.3f})")
    sig = df_vs[df_vs['significant']]
    print(f'\nBG vs CB  (p<0.05: {len(sig)}/{len(df_vs)}):')
    if len(sig)==0: print('  Nenhuma diferença significativa  [n=5, p_min=0.0625]')
    else:
        for _,r in sig.iterrows():
            print(f"  {r['decoder']:12s}  {r['contrast']:14s}  {r['alignment']:6s}"
                  f"  →  {r['winner']}  BG={r['peak_BAC_BG']:.3f}  CB={r['peak_BAC_CB']:.3f}"
                  f"  (p={r['p_value']:.3f})")
    print('='*65+'\n')


# ===========================================================================
# Standard figure helpers
# ===========================================================================

def _ax(ax,wi):
    ax.axhline(CHANCE,ls='--',color='grey',lw=1,alpha=0.5)
    ax.axvline(0,ls='-.',color='grey',lw=1,alpha=0.3)
    ax.set_xlim(wi); ax.set_ylim([0.44,1.0])
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.tick_params(direction='out')

def plot_session_fig(bac_i,bac_r,bm,wi,title,path):
    nc = len(METHODS)
    fig,axes = plt.subplots(2,nc,figsize=(5*nc,7),sharey=True)
    if nc==1: axes=axes.reshape(2,1)
    fig.suptitle(title,fontsize=11,fontweight='bold')
    for col,m in enumerate(METHODS):
        for row,(bac,al) in enumerate([(bac_i,'Init'),(bac_r,'Reach')]):
            ax=axes[row,col]
            for c in CONTRASTS:
                arr=bac.get(m,{}).get(c)
                if arr is None: continue
                ax.plot(bm,arr,color=CONTRAST_COLORS[c],lw=2,label=CONTRAST_LABELS[c])
            _ax(ax,wi); ax.set_title(f'{METHOD_LABELS[m]} — {al}',fontsize=9)
            ax.set_xlabel('time (s)',fontsize=8)
            if col==0: ax.set_ylabel('BAC',fontsize=8)
            if row==0 and col==nc-1: ax.legend(frameon=False,fontsize=8)
    plt.tight_layout(); fig.savefig(path,dpi=150); plt.close(fig)

def plot_mean_fig(mi,si,mr,sr,bm,wi,title,path,n_label=''):
    nc = len(METHODS)
    fig,axes = plt.subplots(2,nc,figsize=(5*nc,7),sharey=True)
    if nc==1: axes=axes.reshape(2,1)
    fig.suptitle(f'{title}  {n_label}',fontsize=11,fontweight='bold')
    for col,m in enumerate(METHODS):
        for row,(means,sems,al) in enumerate([(mi,si,'Init'),(mr,sr,'Reach')]):
            ax=axes[row,col]
            for c in CONTRASTS:
                mn=means.get(m,{}).get(c); sm=sems.get(m,{}).get(c)
                if mn is None: continue
                ax.plot(bm,mn,color=CONTRAST_COLORS[c],lw=2,label=CONTRAST_LABELS[c])
                if sm is not None: ax.fill_between(bm,mn-sm,mn+sm,color=CONTRAST_COLORS[c],alpha=0.15)
            _ax(ax,wi); ax.set_title(f'{METHOD_LABELS[m]} — {al}',fontsize=9)
            ax.set_xlabel('time (s)',fontsize=8)
            if col==0: ax.set_ylabel('BAC',fontsize=8)
            if row==0 and col==nc-1: ax.legend(frameon=False,fontsize=8)
    plt.tight_layout(); fig.savefig(path,dpi=150); plt.close(fig)

def plot_bg_vs_cb(bg_mi,bg_si,bg_mr,bg_sr,cb_mi,cb_si,cb_mr,cb_sr,
                  bm,wi,contrast,path,n_bg,n_cb):
    nc=len(METHODS)
    fig,axes=plt.subplots(2,nc,figsize=(5*nc,7),sharey=True)
    if nc==1: axes=axes.reshape(2,1)
    fig.suptitle(f'BG vs CB — {CONTRAST_LABELS[contrast]}'
                 f'  (BG n={n_bg}, CB n={n_cb})',fontsize=11,fontweight='bold')
    for col,m in enumerate(METHODS):
        for row,(al,gm,get_s) in enumerate([
            ('Init', lambda r,m:(bg_mi if r=='BG' else cb_mi).get(m,{}).get(contrast),
                     lambda r,m:(bg_si if r=='BG' else cb_si).get(m,{}).get(contrast)),
            ('Reach',lambda r,m:(bg_mr if r=='BG' else cb_mr).get(m,{}).get(contrast),
                     lambda r,m:(bg_sr if r=='BG' else cb_sr).get(m,{}).get(contrast)),
        ]):
            ax=axes[row,col]
            for region,color,ls in [('BG','#1F4E79','-'),('CB','#C00000','--')]:
                ma=gm(region,m); sa=get_s(region,m)
                if ma is None: continue
                ax.plot(bm,ma,color=color,lw=2.5,ls=ls,label=region)
                if sa is not None: ax.fill_between(bm,ma-sa,ma+sa,color=color,alpha=0.12)
            _ax(ax,wi); ax.set_title(f'{METHOD_LABELS[m]} — {al}',fontsize=9)
            ax.set_xlabel('time (s)',fontsize=8)
            if col==0: ax.set_ylabel('BAC',fontsize=8)
            if row==0 and col==nc-1: ax.legend(frameon=False,fontsize=8)
    plt.tight_layout(); fig.savefig(path,dpi=150); plt.close(fig)


# ===========================================================================
# MAIN
# ===========================================================================

print('Loading data...')
bg_sessions = load_sessions(BG_NPZ)
cb_sessions = load_sessions(CB_NPZ)
print(f'  BG: {len(bg_sessions)} | CB: {len(cb_sessions)}')

print('\nCollecting BG...')
bg_by_animal,bg_animals,bg_bm,bg_wi = collect_region(bg_sessions,'BG')
print('Collecting CB...')
cb_by_animal,cb_animals,cb_bm,cb_wi = collect_region(cb_sessions,'CB',CB_CELL_TYPE)
bin_mean = bg_bm if bg_bm is not None else cb_bm
win_int  = bg_wi if bg_wi is not None else cb_wi

# ── 1. PER SESSION ────────────────────────────────────────────────────────
print('\n--- Per-session figures ---')
for region,sessions in [('BG',bg_sessions),('CB',cb_sessions)]:
    for sess in sessions:
        animal,s_name = sess['mouse'],sess['sess']
        if is_excluded(animal,s_name,region): continue
        print(f'  [{region}] {animal} {s_name}')
        bi={m:{} for m in METHODS}; br={m:{} for m in METHODS}
        for c in CONTRASTS:
            b0=get_bac(sess,region,c,0,CB_CELL_TYPE)
            b1=get_bac(sess,region,c,1,CB_CELL_TYPE)
            for m in METHODS:
                bi[m][c]=b0.get(m); br[m][c]=b1.get(m)
        plot_session_fig(bi,br,bin_mean,win_int,f'{region} — {animal} | {s_name}',
                         os.path.join(SAVE_DIR,'per_session',region,f'{animal}_{s_name}.png'))

# ── 2. PER ANIMAL ─────────────────────────────────────────────────────────
print('\n--- Per-animal figures ---')
for region,by_animal,animals in [('BG',bg_by_animal,bg_animals),
                                   ('CB',cb_by_animal,cb_animals)]:
    for animal in animals:
        print(f'  [{region}] {animal}')
        mi,si=animal_mean_sem(by_animal,animal,0)
        mr,sr=animal_mean_sem(by_animal,animal,1)
        n=max(len(by_animal[animal][METHODS[0]][CONTRASTS[0]][0]),1)
        plot_mean_fig(mi,si,mr,sr,bin_mean,win_int,f'{region} — {animal}',
                      os.path.join(SAVE_DIR,'per_animal',region,f'{animal}.png'),
                      f'(n={n} sessions)')

# Per-animal BG vs CB — per contrast + all contrasts + pointwise
print('\n--- Per-animal BG vs CB figures ---')
all_animals = sorted(set(bg_animals)|set(cb_animals))
for animal in all_animals:
    print(f'  {animal}')
    # One figure per contrast (with individual sessions)
    for c in CONTRASTS:
        plot_animal_bg_cb(bg_by_animal,cb_by_animal,animal,bin_mean,win_int,c)
    # One figure with all 4 contrasts together
    plot_animal_bg_cb_all_contrasts(bg_by_animal,cb_by_animal,animal,bin_mean,win_int)
    # Pointwise difference CB-BG for all contrasts
    plot_animal_pointwise(bg_by_animal,cb_by_animal,animal,bin_mean,win_int)

# ── 3. GRAND AVERAGE ──────────────────────────────────────────────────────
print('\n--- Grand average ---')
for region,by_animal,animals,bm,wi in [
    ('BG',bg_by_animal,bg_animals,bg_bm,bg_wi),
    ('CB',cb_by_animal,cb_animals,cb_bm,cb_wi)]:
    gmi,gsi,n=grand_mean_sem(by_animal,animals,0)
    gmr,gsr,_ =grand_mean_sem(by_animal,animals,1)
    print(f'  [{region}] n={n}')
    plot_mean_fig(gmi,gsi,gmr,gsr,bm,wi,f'{region} — Grand average',
                  os.path.join(SAVE_DIR,'grand',region,'grand_all.png'),
                  f'(n={n} animals, SEM)')
    for c in CONTRASTS:
        fig,axes=plt.subplots(1,2,figsize=(13,4.5),sharey=True)
        fig.suptitle(f'{region} — {CONTRAST_LABELS[c]} — Grand average (n={n})',
                     fontsize=11,fontweight='bold')
        for align,ax in zip([0,1],axes):
            gm=gmi if align==0 else gmr; gs=gsi if align==0 else gsr
            for m in METHODS:
                mn=gm.get(m,{}).get(c); sm=gs.get(m,{}).get(c)
                if mn is None: continue
                ax.plot(bm,mn,color=METHOD_COLORS[m],lw=2,ls=METHOD_LS[m],
                        label=METHOD_LABELS[m])
                if sm is not None: ax.fill_between(bm,mn-sm,mn+sm,color=METHOD_COLORS[m],alpha=0.15)
            _ax(ax,wi)
            ax.set_xlabel(f'time from {"init" if align==0 else "reach"} (s)',fontsize=9)
            ax.set_ylabel('BAC',fontsize=9); ax.set_title(ALIGN_LABELS[align])
            if align==1: ax.legend(frameon=False,fontsize=9)
        plt.tight_layout()
        fig.savefig(os.path.join(SAVE_DIR,'grand',region,f'grand_{c}.png'),dpi=150)
        plt.close(fig)
    print(f'    saved grand per contrast')

# ── 4. BG vs CB overview ──────────────────────────────────────────────────
print('\n--- BG vs CB ---')
bg_gmi,bg_gsi,n_bg=grand_mean_sem(bg_by_animal,bg_animals,0)
bg_gmr,bg_gsr,_   =grand_mean_sem(bg_by_animal,bg_animals,1)
cb_gmi,cb_gsi,n_cb=grand_mean_sem(cb_by_animal,cb_animals,0)
cb_gmr,cb_gsr,_   =grand_mean_sem(cb_by_animal,cb_animals,1)
for c in CONTRASTS:
    plot_bg_vs_cb(bg_gmi,bg_gsi,bg_gmr,bg_gsr,cb_gmi,cb_gsi,cb_gmr,cb_gsr,
                  bin_mean,win_int,c,
                  os.path.join(SAVE_DIR,'grand',f'BG_vs_CB_{c}.png'),n_bg,n_cb)
# SVM overview
fig,axes=plt.subplots(len(CONTRASTS),2,figsize=(13,4.5*len(CONTRASTS)),sharey=False)
fig.suptitle(f'BG vs CB — SVM — Grand average  (BG n={n_bg}, CB n={n_cb})',
             fontsize=12,fontweight='bold')
for row,c in enumerate(CONTRASTS):
    for col,align in enumerate([0,1]):
        ax=axes[row,col]
        for region,gm,gs,color,ls in [
            ('BG',bg_gmi if align==0 else bg_gmr,bg_gsi if align==0 else bg_gsr,'#1F4E79','-'),
            ('CB',cb_gmi if align==0 else cb_gmr,cb_gsi if align==0 else cb_gsr,'#C00000','--')]:
            mn=gm.get('svm',{}).get(c); sm=gs.get('svm',{}).get(c)
            if mn is None: continue
            ax.plot(bin_mean,mn,color=color,lw=2.5,ls=ls,label=region)
            if sm is not None: ax.fill_between(bin_mean,mn-sm,mn+sm,color=color,alpha=0.12)
        _ax(ax,win_int)
        ax.set_xlabel(f'time from {"init" if align==0 else "reach"} (s)',fontsize=8)
        ax.set_ylabel('BAC',fontsize=8)
        ax.set_title(f'{CONTRAST_LABELS[c]} — {ALIGN_LABELS[align]}',fontsize=9)
        if row==0 and col==1: ax.legend(frameon=False,fontsize=9)
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR,'grand','BG_vs_CB_overview_SVM.png'),dpi=150)
plt.close(fig)
print('  saved BG_vs_CB_overview_SVM.png')

# ── 5. AUC ANALYSIS ───────────────────────────────────────────────────────
print('\n--- AUC analysis ---')
bg_auc = run_auc_analysis(bg_by_animal,bg_animals,'BG',bin_mean)
cb_auc = run_auc_analysis(cb_by_animal,cb_animals,'CB',bin_mean)
bg_auc.to_csv(os.path.join(SAVE_DIR,'auc','auc_BG.csv'),index=False)
cb_auc.to_csv(os.path.join(SAVE_DIR,'auc','auc_CB.csv'),index=False)
print('  Saved auc_BG.csv, auc_CB.csv')
for c in CONTRASTS:
    for a in [0,1]:
        plot_auc_windows(bg_auc,cb_auc,bin_mean,c,a)

# ── 6. POINTWISE ANALYSIS ─────────────────────────────────────────────────
print('\n--- Pointwise analysis ---')
pw_rows = []
for c in CONTRASTS:
    for a in [0,1]:
        res = plot_pointwise(bg_by_animal,bg_animals,cb_by_animal,cb_animals,
                             bin_mean,win_int,c,a)
        if res is None: continue
        # Summary stats for CSV
        n_sig = int(res['sig_mask'].sum())
        pw_rows.append({
            'contrast':CONTRAST_LABELS[c],'alignment':ALIGN_LABELS[a],
            'decoder':METHOD_LABELS[DECODER_PLOT],
            'n_sig_bins_bonferroni':n_sig,
            'pct_sig':round(100*n_sig/len(bin_mean),1),
            'mean_diff_CB_BG':round(float(np.nanmean(res['diff_mean'])),4),
            'max_diff_CB_BG':round(float(np.nanmax(res['diff_mean'])),4),
            'min_diff_CB_BG':round(float(np.nanmin(res['diff_mean'])),4),
            'crossover_times':str([round(x,3) for x in res['crossovers']]),
            'max_cohens_d':round(float(np.nanmax(np.abs(res['d_vals']))),4),
        })
if pw_rows:
    pd.DataFrame(pw_rows).to_csv(
        os.path.join(SAVE_DIR,'stats','pointwise_summary.csv'),index=False)
    print('  Saved pointwise_summary.csv')

# ── 7. LATENCY STATS ──────────────────────────────────────────────────────
print('\n--- Latency analysis ---')
bg_lat = compute_latency_stats(bg_by_animal,bg_animals,'BG',bin_mean)
cb_lat = compute_latency_stats(cb_by_animal,cb_animals,'CB',bin_mean)
lat_all = pd.concat([bg_lat,cb_lat],ignore_index=True)
lat_all.to_csv(os.path.join(SAVE_DIR,'stats','latency_stats.csv'),index=False)
print('  Saved latency_stats.csv')
# Print peak timing summary
print('\n  Peak latency summary (SVM, grand mean across animals):')
for region,lat_df in [('BG',bg_lat),('CB',cb_lat)]:
    sub = lat_df[lat_df['decoder']=='SVM']
    for c in CONTRASTS:
        for a_label in ['Init','Reach']:
            row = sub[(sub['contrast']==CONTRAST_LABELS[c])&
                      (sub['alignment']==a_label)]
            if row.empty: continue
            print(f'  {region:3s}  {CONTRAST_LABELS[c]:14s}  {a_label:6s}'
                  f'  peak={row["peak_time"].mean():.3f}s'
                  f'  BAC={row["peak_BAC"].mean():.3f}'
                  f'  lat_chance={row["latency_above_chance"].mean():.3f}s')

# ── 8. STATISTICS ─────────────────────────────────────────────────────────
print('\n--- Statistics ---')
df_bg   = compute_decoder_stats(bg_by_animal,bg_animals,'BG')
df_cb   = compute_decoder_stats(cb_by_animal,cb_animals,'CB')
df_vs   = compute_region_stats(bg_by_animal,bg_animals,cb_by_animal,cb_animals)
df_pa_bg = compute_per_animal_stats(bg_by_animal,bg_animals,'BG')
df_pa_cb = compute_per_animal_stats(cb_by_animal,cb_animals,'CB')

df_bg.to_csv(os.path.join(SAVE_DIR,'stats','decoder_stats_BG.csv'),index=False)
df_cb.to_csv(os.path.join(SAVE_DIR,'stats','decoder_stats_CB.csv'),index=False)
df_vs.to_csv(os.path.join(SAVE_DIR,'stats','BG_vs_CB_stats.csv'),index=False)
pd.concat([df_pa_bg,df_pa_cb]).to_csv(
    os.path.join(SAVE_DIR,'stats','per_animal_stats.csv'),index=False)
print('  Saved all stats CSVs')
print_summary(df_bg,df_cb,df_vs)

print(f'\nAll done! → {SAVE_DIR}')

## 5. Robustness analyses — `per_animal_robustness.py`
Neuron-count control, neuron-dropping, and cross-session reliability analyses.

In [ ]:
"""
per_animal_robustness.py
========================
Três análises per-animal para reforçar a robustez do resultado CB > BG:

  1. NEURON COUNT CONTROL
     Scatter peak BAC vs n_neurons por sessão, por região, regressão por animal.
     → A vantagem do CB sobrevive ao controlo do número de neurónios?

  2. NEURON-DROPPING CURVES
     Decoding com k = 5,10,20,40,80,... neurónios (subsampling aleatório).
     → Cada neurónio do CB carrega mais informação? Comparação a k igualado.

  3. CROSS-SESSION RELIABILITY
     Correlação das curvas BAC entre pares de sessões dentro de cada animal.
     → Quão reproduzível é o padrão temporal? (formaliza a variabilidade da Milka)

As análises 1 e 3 usam os BAC results dos npz (rápido).
A análise 2 (neuron-dropping) re-carrega os .mat porque precisa dos
spike counts raw — é a mais lenta.

Outputs (SAVE_DIR/):
  neuron_count/scatter_{contrast}_{align}.png
  neuron_count/neuron_count_stats.csv
  neuron_dropping/{animal}_{region}_dropping_{contrast}_{align}.png
  neuron_dropping/dropping_summary.png
  neuron_dropping/dropping_stats.csv
  reliability/{animal}_reliability_matrix.png
  reliability/reliability_stats.csv
"""

import os
import time
import numpy as np
import pandas as pd
import scipy.io as sio
import h5py
import matplotlib.pyplot as plt
from scipy import stats
from joblib import Parallel, delayed
from sklearn.svm import SVC

# ===========================================================================
# Configuration
# ===========================================================================
rootdir  = r'path/to/ephys_data'  # ephys_and_behavior/mat_files directory
group    = '20230801_ChocolateGroup'
base_mat = os.path.join(rootdir,'ephys_and_behavior','mat_files',group)

BG_NPZ   = r'data/SVM_sess_BG.npz'
CB_NPZ   = r'data/SVM_sess_CB.npz'
SAVE_DIR = r'output/robustness'

METHOD        = 'svm'
METHOD_LABEL  = 'SVM'
CONTRASTS     = ['pp','D','C','nD']
CONTRAST_LABELS = {'pp':'Push / Pull','D':'Dominant','C':'Center','nD':'Non-dominant'}
ALIGN_LABELS  = {0:'Init',1:'Reach'}
REGION_COLORS = {'BG':'#1F4E79','CB':'#C00000'}
CHANCE        = 0.5

# Neuron-dropping parameters
DROP_SIZES    = [5,10,20,40,80,160]   # subsample sizes
N_SUBSAMPLES  = 20                     # random draws per size
DROP_NCV      = 30                     # CV repeats per draw (lower — many draws)
DROP_CONTRAST = 'pp'                   # contrast for neuron-dropping (strongest)
DROP_ALIGN    = 1                      # 0=Init, 1=Reach
PEAK_WINDOW   = (-0.5, 1.0)            # window to find peak (s) for dropping

CB_CELL_TYPE  = 'all'
MIN_TRIALS    = 10
RATIO_TV      = 0.8

# Sliding window
win_int   = [-2.5,2.5]
bin_width = 0.1
bin_step  = 0.025
tm_before,tm_after = win_int
n_bins    = int(np.floor(((tm_after-tm_before)/bin_step)-(bin_width/bin_step))+1)
bin_edges = np.array([[tm_before+i*bin_step, tm_before+bin_width+i*bin_step]
                       for i in range(n_bins)])
bin_mean  = bin_edges[:,0]+bin_width/2

ANIMALS = ['CoteDor','Lindt','Toblerone','Milka','FerreroRocher']
EXCLUSIONS = [
    ('3_Toblerone','R2','both'),('3_Toblerone','R3','both'),
    ('3_Toblerone','R4','both'),('3_Toblerone','R6','both'),
    ('4_Milka','R7','both'),
]

for sub in ['neuron_count','neuron_dropping','reliability']:
    os.makedirs(os.path.join(SAVE_DIR,sub),exist_ok=True)

def is_excl(animal,sess,region):
    for ea,es,er in EXCLUSIONS:
        if animal==ea and sess==es and (er=='both' or er==region):
            return True
    return False

def load_sessions(p):
    return np.load(p,allow_pickle=True)['SVM_sess'].tolist()


# ===========================================================================
# Loaders for raw spikes (neuron-dropping)
# ===========================================================================

def _arr(v): return np.atleast_1d(np.squeeze(np.asarray(v))).ravel()
def _load_bhv(p): return sio.loadmat(p,simplify_cells=True)['bhv']
def _n_neu(p):
    with h5py.File(p,'r') as f: return f['eg_neurons']['st_init'].shape[1]
def _load_spikes(p,n):
    si,sr=[],[]
    with h5py.File(p,'r') as f:
        eg=f['eg_neurons']
        for k in range(n):
            di=f[eg['st_init'][0,k]]; dr=f[eg['st_reach'][0,k]]
            si.append([f[di[t,0]][()].ravel().astype(float) for t in range(di.shape[0])])
            sr.append([f[dr[t,0]][()].ravel().astype(float) for t in range(dr.shape[0])])
    return si,sr
def _build_sc(st,nt,nn):
    sc=np.zeros((n_bins,nt,nn),dtype=np.float32)
    for ni in range(nn):
        for tt in range(nt):
            spk=np.atleast_1d(st[ni][tt]).ravel()
            for i,(bs,be) in enumerate(bin_edges):
                sc[i,tt,ni]=np.sum((spk>bs)&(spk<be))
    return sc


# ===========================================================================
# Get n_neurons per session from .mat
# ===========================================================================

def get_n_neurons(sess,region):
    mouse,s_name = sess['mouse'],sess['sess']
    ep = os.path.join(base_mat,mouse,s_name,f'eg_neurons_{region}.mat')
    if not os.path.isfile(ep): return None
    try:    return _n_neu(ep)
    except: return None


# ===========================================================================
# ANALYSIS 1 — Neuron count control
# ===========================================================================

def get_peak_bac(sess,region,contrast,align,cb_ct='all'):
    bac = sess.get('bac',{})
    try:
        arr = np.array(bac[METHOD][contrast] if region=='BG'
                       else bac[cb_ct][METHOD][contrast])
        return float(np.nanmax(arr[:,align]))
    except:
        return None

def run_neuron_count(bg_sessions,cb_sessions):
    print('\n=== 1. NEURON COUNT CONTROL ===')
    rows = []
    for region,sessions in [('BG',bg_sessions),('CB',cb_sessions)]:
        for sess in sessions:
            animal,s_name = sess['mouse'],sess['sess']
            if is_excl(animal,s_name,region): continue
            n_neu = get_n_neurons(sess,region)
            if n_neu is None: continue
            for c in CONTRASTS:
                for a in [0,1]:
                    pk = get_peak_bac(sess,region,c,a,CB_CELL_TYPE)
                    if pk is None: continue
                    rows.append({'region':region,'animal':animal,'session':s_name,
                                 'n_neurons':n_neu,'contrast':CONTRAST_LABELS[c],
                                 'alignment':ALIGN_LABELS[a],'peak_BAC':round(pk,4)})
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(SAVE_DIR,'neuron_count','neuron_count_stats.csv'),index=False)
    print(f'  Saved neuron_count_stats.csv ({len(df)} rows)')

    # Scatter per contrast × align
    for c in CONTRASTS:
        for a in [0,1]:
            sub = df[(df['contrast']==CONTRAST_LABELS[c]) &
                     (df['alignment']==ALIGN_LABELS[a])]
            if sub.empty: continue
            fig,ax = plt.subplots(figsize=(8,6))
            for region in ['BG','CB']:
                rs = sub[sub['region']==region]
                if rs.empty: continue
                ax.scatter(rs['n_neurons'],rs['peak_BAC'],
                           color=REGION_COLORS[region],alpha=0.6,s=40,label=region)
                # Regression line
                if len(rs)>=3:
                    x=rs['n_neurons'].values; y=rs['peak_BAC'].values
                    valid=~(np.isnan(x)|np.isnan(y))
                    if valid.sum()>=3:
                        slope,inter,r,p,_ = stats.linregress(x[valid],y[valid])
                        xs=np.linspace(x[valid].min(),x[valid].max(),50)
                        ax.plot(xs,slope*xs+inter,color=REGION_COLORS[region],
                                lw=2,ls='--',alpha=0.8,
                                label=f'{region} fit (r={r:.2f}, p={p:.3f})')
            ax.axhline(CHANCE,ls=':',color='grey',alpha=0.5)
            ax.set_xlabel('Number of neurons',fontsize=10)
            ax.set_ylabel('Peak BAC',fontsize=10)
            ax.set_title(f'Peak BAC vs neuron count — {CONTRAST_LABELS[c]} — {ALIGN_LABELS[a]}',
                         fontsize=11,fontweight='bold')
            ax.legend(frameon=False,fontsize=8)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            plt.tight_layout()
            fig.savefig(os.path.join(SAVE_DIR,'neuron_count',
                                     f'scatter_{c}_{ALIGN_LABELS[a]}.png'),dpi=150)
            plt.close(fig)
    print('  Saved scatter plots')

    # Summary: median n_neurons per region
    print('\n  Median neuron count:')
    for region in ['BG','CB']:
        med = df[df['region']==region]['n_neurons'].median()
        print(f'    {region}: {med:.0f} neurons')
    return df


# ===========================================================================
# ANALYSIS 2 — Neuron-dropping curves
# ===========================================================================

def _decode_subset(spk, labels, neuron_idx, ncv=DROP_NCV):
    """Decode using only neuron_idx columns, at peak window. Returns peak BAC."""
    labels = np.asarray(labels,dtype=int)
    n_trials = len(labels)
    ntrain = int(np.floor(n_trials*RATIO_TV))
    cls,cts = np.unique(labels,return_counts=True)
    if len(cls)<2 or cts.min()<MIN_TRIALS: return np.nan

    # Peak window bins
    pw = (bin_mean>=PEAK_WINDOW[0]) & (bin_mean<=PEAK_WINDOW[1])
    pw_idx = np.where(pw)[0]

    bac_per_bin = np.full(len(pw_idx), np.nan)

    for bi,b in enumerate(pw_idx):
        X = spk[b][:,neuron_idx].astype(float)
        cv_bacs = []
        for _ in range(ncv):
            rp = np.random.permutation(n_trials)
            tri,vai = rp[:ntrain],rp[ntrain:]
            if len(np.unique(labels[tri]))<2 or len(np.unique(labels[vai]))<2:
                continue
            Xtr,Xva = X[tri],X[vai]
            mn=np.mean(Xtr,axis=0); mx=np.max(Xtr,axis=0)
            den=np.where(mx==0,1.0,mx)
            Xtr=(Xtr-mn)/den; Xva=(Xva-mn)/den
            try:
                clf=SVC(kernel='linear',C=1.0); clf.fit(Xtr,labels[tri])
                pred=clf.predict(Xva)
                yt=labels[vai]
                tp=np.sum((pred==1)&(yt==1)); tn=np.sum((pred==0)&(yt==0))
                fp=np.sum((pred==1)&(yt==0)); fn=np.sum((pred==0)&(yt==1))
                s=tp/(tp+fn) if (tp+fn)>0 else np.nan
                q=tn/(tn+fp) if (tn+fp)>0 else np.nan
                cv_bacs.append(np.nanmean([s,q]))
            except: pass
        if cv_bacs: bac_per_bin[bi]=np.nanmean(cv_bacs)
    return np.nanmax(bac_per_bin) if not np.all(np.isnan(bac_per_bin)) else np.nan


def get_spk_labels(sess,region,contrast,align):
    mouse,s_name = sess['mouse'],sess['sess']
    ep=os.path.join(base_mat,mouse,s_name,f'eg_neurons_{region}.mat')
    bp=os.path.join(base_mat,mouse,s_name,'behavior_fundamentals.mat')
    if not os.path.isfile(ep) or not os.path.isfile(bp): return None,None
    bhv=_load_bhv(bp)
    inv_pp=_arr(bhv['init_invPush_invPull_idx']).astype(int)
    i_pp=np.where(inv_pp==1,0,np.where(inv_pp==2,1,3))
    i_DCnD=_arr(bhv['DCnD_init']).astype(int)
    r_pp=_arr(bhv['pp_inVec_idx']).astype(int)
    r_DCnD=_arr(bhv['DCnD_inVec_idx']).astype(int)
    r_hit=np.nan_to_num(np.atleast_1d(bhv['hit_inVec']).ravel().astype(float),nan=0.).astype(int)
    nn=_n_neu(ep)
    si,sr=_load_spikes(ep,nn)
    sc_i=_build_sc(si,len(i_pp),nn); sc_r=_build_sc(sr,len(r_hit),nn)
    hit=r_hit==1
    idx={('pp',0):(i_pp==0,i_pp==1,sc_i),
         ('D',0):(i_DCnD==1,(i_DCnD==2)|(i_DCnD==3),sc_i),
         ('C',0):(i_DCnD==2,(i_DCnD==1)|(i_DCnD==3),sc_i),
         ('nD',0):(i_DCnD==3,(i_DCnD==1)|(i_DCnD==2),sc_i),
         ('pp',1):((r_pp==0)&hit,(r_pp==1)&hit,sc_r),
         ('D',1):((r_DCnD==1)&hit,((r_DCnD==2)|(r_DCnD==3))&hit,sc_r),
         ('C',1):((r_DCnD==2)&hit,((r_DCnD==1)|(r_DCnD==3))&hit,sc_r),
         ('nD',1):((r_DCnD==3)&hit,((r_DCnD==1)|(r_DCnD==2))&hit,sc_r)}
    m1,m2,sc=idx[(contrast,align)]
    s1,s2=sc[:,m1,:],sc[:,m2,:]
    if s1.shape[1]<MIN_TRIALS or s2.shape[1]<MIN_TRIALS: return None,None
    spk=np.concatenate([s1,s2],axis=1)
    labels=np.array([0]*s1.shape[1]+[1]*s2.shape[1])
    return spk,labels


def run_neuron_dropping(bg_sessions,cb_sessions):
    print('\n=== 2. NEURON-DROPPING CURVES ===')
    print(f'  Contrast={DROP_CONTRAST}, Align={ALIGN_LABELS[DROP_ALIGN]}')
    rows = []

    for region,sessions in [('BG',bg_sessions),('CB',cb_sessions)]:
        for sess in sessions:
            animal,s_name = sess['mouse'],sess['sess']
            if is_excl(animal,s_name,region): continue
            spk,labels = get_spk_labels(sess,region,DROP_CONTRAST,DROP_ALIGN)
            if spk is None: continue
            n_avail = spk.shape[2]
            print(f'  [{region}] {animal} {s_name}: {n_avail} neurons',flush=True)

            for k in DROP_SIZES:
                if k > n_avail: continue
                # N_SUBSAMPLES random draws of k neurons
                draw_bacs = Parallel(n_jobs=-1,prefer='threads')(
                    delayed(_decode_subset)(
                        spk,labels,
                        np.random.choice(n_avail,k,replace=False)
                    ) for _ in range(N_SUBSAMPLES)
                )
                rows.append({'region':region,'animal':animal,'session':s_name,
                             'k_neurons':k,'n_available':n_avail,
                             'mean_peak_BAC':round(float(np.nanmean(draw_bacs)),4),
                             'sem_peak_BAC':round(float(np.nanstd(draw_bacs)/np.sqrt(len(draw_bacs))),4)})

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(SAVE_DIR,'neuron_dropping','dropping_stats.csv'),index=False)
    print(f'  Saved dropping_stats.csv ({len(df)} rows)')

    # Summary plot: BAC vs k, BG vs CB (averaged across animals/sessions)
    fig,ax = plt.subplots(figsize=(9,6))
    for region in ['BG','CB']:
        rs = df[df['region']==region]
        if rs.empty: continue
        ks,means,sems = [],[],[]
        for k in DROP_SIZES:
            sub = rs[rs['k_neurons']==k]
            if sub.empty: continue
            ks.append(k); means.append(sub['mean_peak_BAC'].mean())
            sems.append(sub['mean_peak_BAC'].sem())
        ax.errorbar(ks,means,yerr=sems,color=REGION_COLORS[region],
                    lw=2,marker='o',capsize=4,label=region)
    ax.axhline(CHANCE,ls=':',color='grey',alpha=0.5,label='chance')
    ax.set_xlabel('Number of neurons (random subsample)',fontsize=10)
    ax.set_ylabel('Peak BAC',fontsize=10)
    ax.set_xscale('log'); ax.set_xticks(DROP_SIZES); ax.set_xticklabels(DROP_SIZES)
    ax.set_title(f'Neuron-dropping curve — {CONTRAST_LABELS[DROP_CONTRAST]} — {ALIGN_LABELS[DROP_ALIGN]}'
                 '\n(steeper = more information per neuron)',
                 fontsize=11,fontweight='bold')
    ax.legend(frameon=False,fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR,'neuron_dropping','dropping_summary.png'),dpi=150)
    plt.close(fig)
    print('  Saved dropping_summary.png')

    # Per-animal dropping curves
    for animal in df['animal'].unique():
        fig,ax = plt.subplots(figsize=(9,6))
        for region in ['BG','CB']:
            rs = df[(df['animal']==animal)&(df['region']==region)]
            if rs.empty: continue
            ks,means,sems=[],[],[]
            for k in DROP_SIZES:
                sub=rs[rs['k_neurons']==k]
                if sub.empty: continue
                ks.append(k); means.append(sub['mean_peak_BAC'].mean())
                sems.append(sub['sem_peak_BAC'].mean())
            ax.errorbar(ks,means,yerr=sems,color=REGION_COLORS[region],
                        lw=2,marker='o',capsize=4,label=region)
        ax.axhline(CHANCE,ls=':',color='grey',alpha=0.5)
        ax.set_xlabel('Number of neurons',fontsize=10)
        ax.set_ylabel('Peak BAC',fontsize=10)
        ax.set_xscale('log'); ax.set_xticks(DROP_SIZES); ax.set_xticklabels(DROP_SIZES)
        ax.set_title(f'{animal} — Neuron-dropping — {CONTRAST_LABELS[DROP_CONTRAST]} {ALIGN_LABELS[DROP_ALIGN]}',
                     fontsize=11,fontweight='bold')
        ax.legend(frameon=False,fontsize=9)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        plt.tight_layout()
        fig.savefig(os.path.join(SAVE_DIR,'neuron_dropping',
                                 f'{animal}_dropping.png'),dpi=150)
        plt.close(fig)
    print('  Saved per-animal dropping curves')
    return df


# ===========================================================================
# ANALYSIS 3 — Cross-session reliability
# ===========================================================================

def get_bac_curve(sess,region,contrast,align,cb_ct='all'):
    bac=sess.get('bac',{})
    try:
        arr=np.array(bac[METHOD][contrast] if region=='BG'
                     else bac[cb_ct][METHOD][contrast])
        return arr[:,align]
    except:
        return None

def run_reliability(bg_sessions,cb_sessions):
    print('\n=== 3. CROSS-SESSION RELIABILITY ===')
    rows = []

    for region,sessions in [('BG',bg_sessions),('CB',cb_sessions)]:
        # Group sessions by animal
        by_animal = {}
        for sess in sessions:
            animal,s_name = sess['mouse'],sess['sess']
            if is_excl(animal,s_name,region): continue
            by_animal.setdefault(animal,[]).append(sess)

        for animal,sess_list in by_animal.items():
            if len(sess_list)<2: continue
            for c in CONTRASTS:
                for a in [0,1]:
                    curves = []
                    for sess in sess_list:
                        cv = get_bac_curve(sess,region,c,a,CB_CELL_TYPE)
                        if cv is not None and not np.all(np.isnan(cv)):
                            curves.append(cv)
                    if len(curves)<2: continue
                    # Pairwise correlations between sessions
                    corrs = []
                    for i in range(len(curves)):
                        for j in range(i+1,len(curves)):
                            v = ~(np.isnan(curves[i])|np.isnan(curves[j]))
                            if v.sum()>10:
                                r,_ = stats.pearsonr(curves[i][v],curves[j][v])
                                corrs.append(r)
                    if corrs:
                        rows.append({'region':region,'animal':animal,
                                     'contrast':CONTRAST_LABELS[c],
                                     'alignment':ALIGN_LABELS[a],
                                     'n_sessions':len(curves),
                                     'n_pairs':len(corrs),
                                     'mean_reliability':round(float(np.mean(corrs)),4),
                                     'min_reliability':round(float(np.min(corrs)),4),
                                     'max_reliability':round(float(np.max(corrs)),4)})

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(SAVE_DIR,'reliability','reliability_stats.csv'),index=False)
    print(f'  Saved reliability_stats.csv ({len(df)} rows)')

    # Heatmap: animal × contrast, mean reliability (one per region/align)
    for region in ['BG','CB']:
        for a_label in ['Init','Reach']:
            sub = df[(df['region']==region)&(df['alignment']==a_label)]
            if sub.empty: continue
            animals = sorted(sub['animal'].unique())
            contrasts = [CONTRAST_LABELS[c] for c in CONTRASTS]
            mat = np.full((len(animals),len(contrasts)),np.nan)
            for i,an in enumerate(animals):
                for j,cl in enumerate(contrasts):
                    row = sub[(sub['animal']==an)&(sub['contrast']==cl)]
                    if not row.empty:
                        mat[i,j] = row['mean_reliability'].values[0]
            fig,ax = plt.subplots(figsize=(8,5))
            im = ax.imshow(mat,cmap='RdYlGn',vmin=0,vmax=1,aspect='auto')
            ax.set_xticks(range(len(contrasts))); ax.set_xticklabels(contrasts,rotation=30,ha='right')
            ax.set_yticks(range(len(animals))); ax.set_yticklabels(animals)
            for i in range(len(animals)):
                for j in range(len(contrasts)):
                    if not np.isnan(mat[i,j]):
                        ax.text(j,i,f'{mat[i,j]:.2f}',ha='center',va='center',fontsize=9)
            ax.set_title(f'Cross-session reliability — {region} — {a_label}'
                         '\n(Pearson r between session BAC curves)',
                         fontsize=11,fontweight='bold')
            plt.colorbar(im,ax=ax,label='mean reliability (r)',fraction=0.046)
            plt.tight_layout()
            fig.savefig(os.path.join(SAVE_DIR,'reliability',
                                     f'reliability_{region}_{a_label}.png'),dpi=150)
            plt.close(fig)
    print('  Saved reliability heatmaps')

    # Print summary
    print('\n  Mean reliability by animal (SVM, all contrasts/aligns):')
    for animal in sorted(df['animal'].unique()):
        for region in ['BG','CB']:
            sub = df[(df['animal']==animal)&(df['region']==region)]
            if sub.empty: continue
            print(f'    {animal:16s} {region}: r = {sub["mean_reliability"].mean():.3f}')
    return df


# ===========================================================================
# MAIN
# ===========================================================================

print('Loading sessions...')
bg_sessions = load_sessions(BG_NPZ)
cb_sessions = load_sessions(CB_NPZ)
print(f'  BG: {len(bg_sessions)} | CB: {len(cb_sessions)}')

t0 = time.time()

# Analysis 1 — fast (uses npz)
df_count = run_neuron_count(bg_sessions,cb_sessions)

# Analysis 3 — fast (uses npz)
df_reli  = run_reliability(bg_sessions,cb_sessions)

# Analysis 2 — slow (re-loads .mat)
df_drop  = run_neuron_dropping(bg_sessions,cb_sessions)

print(f'\nAll done in {time.time()-t0:.0f}s → {SAVE_DIR}')